In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')
# Enveda CASMI 2026 — Kaggle Notebook starter
from pathlib import Path
import glob
import numpy as np
import pandas as pd

# Cell 1: locate competition files
test_paths = glob.glob("/kaggle/input/**/test.parquet", recursive=True)
print("Found test files:", test_paths)
if not test_paths:
    raise FileNotFoundError("Attach the competition dataset to this Kaggle Notebook.")
DATA = Path(test_paths[0]).parent
TRAIN_PATH, TEST_PATH = DATA/"train.parquet", DATA/"test.parquet"
SAMPLE_PATH = DATA/"sample_submission.csv"

# Cell 2: inspect test and submission schema
test = pd.read_parquet(TEST_PATH)
sample = pd.read_csv(SAMPLE_PATH)
print("Test shape:", test.shape)
print("Test columns:", test.columns.tolist())
print("Spectra per molecule summary:")
print(test.groupby("molecule_id").size().describe())
print("Sample submission:", sample.shape, sample.columns.tolist())
print("Adduct counts:")
print(test["adduct"].value_counts(dropna=False))
print("Ionization mode counts:")
print(test["ionization_mode"].value_counts(dropna=False))

# Cell 3: read a small, deterministic slice of training data to smoke-test the pipeline.
# Once this works, increase the slice or build a vectorized index over the full dataset.
TRAIN_ROWS_FOR_SMOKE_TEST = 50000
# Read only a few Parquet row groups instead of materializing the entire ~3 GB training file.
import pyarrow.parquet as pq
pf = pq.ParquetFile(TRAIN_PATH)
print("Train row groups:", pf.num_row_groups, "| total rows:", pf.metadata.num_rows)
needed = ["precursor_mz", "adduct", "normalized_smiles", "inchikey14"]
if "ingest_lib" in pf.schema.names:
    needed.append("ingest_lib")
chunks, rows_read = [], 0
for rg in range(pf.num_row_groups):
    chunk = pf.read_row_group(rg, columns=needed).to_pandas()
    chunks.append(chunk)
    rows_read += len(chunk)
    if rows_read >= TRAIN_ROWS_FOR_SMOKE_TEST:
        break
train_small = pd.concat(chunks, ignore_index=True).head(TRAIN_ROWS_FOR_SMOKE_TEST)
print("Loaded training smoke-test rows:", train_small.shape)
print("Top source libraries:")
if "ingest_lib" in train_small:
    print(train_small["ingest_lib"].value_counts(dropna=False).head(10))
print("Train sample:")
print(train_small[["precursor_mz", "adduct", "normalized_smiles", "inchikey14"]].head())

# Cell 4: small precursor-mass retrieval baseline.
# This ranks training structures with nearby precursor m/z. It is deliberately a simple
# benchmark to validate output plumbing, not a final spectral similarity model.
train_small = train_small.dropna(subset=["precursor_mz", "normalized_smiles"])
mass = train_small["precursor_mz"].to_numpy(dtype=float)
smiles = train_small["normalized_smiles"].astype(str).to_numpy()
keys = train_small["inchikey14"].astype(str).to_numpy() if "inchikey14" in train_small else smiles

# Candidate ranking uses nearest precursor m/z, with duplicate structures deduplicated.
# Spectrum-aware scoring is the next upgrade.
submission_rows = []
for n, molecule_id in enumerate(sample["molecule_id"]):
    group = test[test["molecule_id"] == molecule_id]
    scores = {}
    for row in group.itertuples(index=False):
        distance = np.abs(mass - float(row.precursor_mz))
        nearby = np.flatnonzero(distance <= 0.05)
        if len(nearby) == 0:
            nearby = np.argsort(distance)[:100]
        # Keep each structure's best mass match across all spectra for this molecule.
        for j in nearby:
            smi = smiles[j]
            score = -float(distance[j])
            if smi not in scores or score > scores[smi]:
                scores[smi] = score
    ranked = [s for s, _ in sorted(scores.items(), key=lambda kv: kv[1], reverse=True)[:25]]
    if not ranked:
        ranked = ["CCO"]  # valid-format fallback only; should be rare and is not a chemistry prediction
    submission_rows.append({"molecule_id": molecule_id, "smiles": ";".join(ranked)})
    if (n + 1) % 50 == 0:
        print(f"Processed {n+1}/{len(sample)} molecules")

submission = pd.DataFrame(submission_rows)
assert submission.columns.tolist() == ["molecule_id", "smiles"]
assert submission["molecule_id"].is_unique
assert submission["smiles"].notna().all() and (submission["smiles"].str.len() > 0).all()
assert (submission["smiles"].str.split(";").str.len() <= 25).all()
out = "/kaggle/working/submission_baseline.csv"
submission.to_csv(out, index=False)
print("Wrote", out, "shape=", submission.shape)
print(submission.head().to_string(index=False))


In [ ]:
import pandas as pd
from pathlib import Path

submission_path = Path("/kaggle/working/submission_baseline.csv")
sample_path = Path(
    "/kaggle/input/datasets/denisketer/enveda-casmi-2026/sample_submission.csv"
)

assert submission_path.exists(), "Submission file not found!"

sub = pd.read_csv(submission_path)
sample = pd.read_csv(sample_path)

print("Submission shape:", sub.shape)
print("Expected shape:", sample.shape)
print("Columns:", sub.columns.tolist())
print("Unique molecules:", sub["molecule_id"].nunique())
print("Missing values:\n", sub.isna().sum())

# Check whether each row has up to 25 candidate SMILES
candidate_counts = sub["smiles"].fillna("").apply(
    lambda x: len([s for s in x.split(";") if s.strip()])
)
print("\nCandidate counts:")
print(candidate_counts.describe())
print("Rows with zero candidates:", (candidate_counts == 0).sum())

assert sub.columns.tolist() == sample.columns.tolist()
assert sub["molecule_id"].is_unique
assert len(sub) == len(sample)
assert (candidate_counts <= 25).all()

print("\nBasic submission checks passed.")

In [ ]:
import pyarrow.parquet as pq
from pathlib import Path

train_path = Path(
    "/kaggle/input/datasets/denisketer/enveda-casmi-2026/train.parquet"
)

parquet_file = pq.ParquetFile(train_path)

print("Training rows:", parquet_file.metadata.num_rows)
print("Row groups:", parquet_file.num_row_groups)
print("\nTraining schema:")
print(parquet_file.schema)

In [ ]:
import pyarrow.parquet as pq
import pandas as pd
from pathlib import Path

train_path = Path(
    "/kaggle/input/datasets/denisketer/enveda-casmi-2026/train.parquet"
)

# Read only the columns needed for an initial inspection
columns = [
    "normalized_smiles",
    "inchikey14",
    "molecular_formula",
    "precursor_mz",
    "adduct",
    "ionization_mode",
    "ingest_lib",
]

pf = pq.ParquetFile(train_path)

total_rows = 0
missing_counts = {c: 0 for c in columns}
adduct_counts = {}
library_counts = {}
unique_keys = set()
mass_min = float("inf")
mass_max = float("-inf")

for batch in pf.iter_batches(
    batch_size=100_000,
    columns=columns
):
    df = batch.to_pandas()
    total_rows += len(df)

    for col in columns:
        missing_counts[col] += int(df[col].isna().sum())

    # Count common adducts and source libraries
    for key, count in df["adduct"].value_counts(
        dropna=False
    ).items():
        adduct_counts[key] = adduct_counts.get(key, 0) + int(count)

    for key, count in df["ingest_lib"].value_counts(
        dropna=False
    ).items():
        library_counts[key] = library_counts.get(key, 0) + int(count)

    # Count distinct labelled structures
    valid_keys = df["inchikey14"].dropna().unique()
    unique_keys.update(valid_keys)

    # Track the precursor mass range
    masses = df["precursor_mz"].dropna()
    if not masses.empty:
        mass_min = min(mass_min, masses.min())
        mass_max = max(mass_max, masses.max())

print("Rows inspected:", total_rows)
print("Distinct InChIKey14 labels:", len(unique_keys))
print("Precursor m/z range:", mass_min, "to", mass_max)

print("\nMissing values:")
print(pd.Series(missing_counts).sort_values(ascending=False))

print("\nMost common adducts:")
print(pd.Series(adduct_counts).sort_values(ascending=False).head(10))

print("\nSource libraries:")
print(pd.Series(library_counts).sort_values(ascending=False))

In [ ]:
import pyarrow.parquet as pq
import pandas as pd
import numpy as np

train_path = (
    "/kaggle/input/datasets/denisketer/"
    "enveda-casmi-2026/train.parquet"
)

pf = pq.ParquetFile(train_path)

batches = []

for batch in pf.iter_batches(
    batch_size=100_000,
    columns=["precursor_mz", "adduct", "ingest_lib"]
):
    batches.append(batch.to_pandas())

mass_df = pd.concat(batches, ignore_index=True)

print("Overall precursor m/z quantiles:")
print(
    mass_df["precursor_mz"].quantile(
        [0, 0.001, 0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99, 0.999, 1]
    )
)

print("\nLowest 10 precursor masses:")
print(
    mass_df.nsmallest(10, "precursor_mz").to_string(index=False)
)

print("\nHighest 10 precursor masses:")
print(
    mass_df.nlargest(10, "precursor_mz").to_string(index=False)
)

print("\nMass quantiles by common adduct:")
for adduct, group in mass_df.groupby("adduct"):
    if len(group) >= 1000:
        print(
            adduct,
            "count =", len(group),
            "median =", round(group["precursor_mz"].median(), 3),
            "99th percentile =", round(
                group["precursor_mz"].quantile(0.99), 3
            )
        )

print("\nRows below m/z 50:", (mass_df["precursor_mz"] < 50).sum())
print("Rows above m/z 2000:", (mass_df["precursor_mz"] > 2000).sum())

In [ ]:
import pyarrow.parquet as pq
import pandas as pd

train_path = (
    "/kaggle/input/datasets/denisketer/"
    "enveda-casmi-2026/train.parquet"
)

pf = pq.ParquetFile(train_path)

columns = [
    "precursor_mz",
    "adduct",
    "ingest_lib",
    "normalized_smiles",
    "molecular_formula",
    "inchikey14",
    "precursor_error_ppm",
    "num_peaks",
]

outliers = []

for batch in pf.iter_batches(
    batch_size=100_000,
    columns=columns
):
    df = batch.to_pandas()

    unusual = df[
        (df["precursor_mz"] < 50) |
        (df["precursor_mz"] > 2000)
    ]

    if not unusual.empty:
        outliers.append(unusual)

outliers = pd.concat(outliers, ignore_index=True)

print("Number of unusual records:", len(outliers))

print(
    outliers[
        [
            "precursor_mz",
            "adduct",
            "ingest_lib",
            "molecular_formula",
            "precursor_error_ppm",
            "num_peaks",
            "inchikey14",
        ]
    ].sort_values("precursor_mz").to_string(index=False)
)

In [ ]:
try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors, rdMolDescriptors

    print("RDKit is available.")

    mol = Chem.MolFromSmiles("CCO")
    print("Test molecule: ethanol")
    print("Molecular formula:", rdMolDescriptors.CalcMolFormula(mol))
    print("Molecular weight:", round(Descriptors.MolWt(mol), 4))

except ImportError:
    print("RDKit is not installed in this environment.")

In [ ]:
%pip install rdkit

In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors

mol = Chem.MolFromSmiles("CCO")

print("RDKit version:", __import__("rdkit").__version__)
print("SMILES valid:", mol is not None)
print("Molecular formula:", rdMolDescriptors.CalcMolFormula(mol))
print("Exact molecular mass:", round(Descriptors.ExactMolWt(mol), 5))

In [ ]:
try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors, rdMolDescriptors

    print("RDKit is available.")

    mol = Chem.MolFromSmiles("CCO")
    print("Test molecule: ethanol")
    print("Molecular formula:", rdMolDescriptors.CalcMolFormula(mol))
    print("Molecular weight:", round(Descriptors.MolWt(mol), 4))

except ImportError:
    print("RDKit is not installed in this environment.")

In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors

mol = Chem.MolFromSmiles("CCO")

print("Molecular formula:", rdMolDescriptors.CalcMolFormula(mol))
print("Average molecular weight:", round(Descriptors.MolWt(mol), 5))
print("Exact molecular mass:", round(Descriptors.ExactMolWt(mol), 5))

In [ ]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors

DATA_DIR = "/kaggle/input/datasets/denisketer/enveda-casmi-2026"

# Read only the columns needed to construct the structure library.
train = pd.read_parquet(
    f"{DATA_DIR}/train.parquet",
    columns=[
        "normalized_smiles",
        "inchikey14",
        "molecular_formula",
    ],
)

# One row per InChIKey14, retaining a representative SMILES.
library = train.drop_duplicates("inchikey14").copy()
library = library.dropna(subset=["normalized_smiles", "inchikey14"])

def describe_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return {
        "exact_mass": Descriptors.ExactMolWt(mol),
        "rdkit_formula": rdMolDescriptors.CalcMolFormula(mol),
    }

descriptions = library["normalized_smiles"].map(describe_smiles)
valid = descriptions.notna()

library = library.loc[valid].copy()
library["exact_mass"] = [
    item["exact_mass"] for item in descriptions.loc[valid]
]
library["rdkit_formula"] = [
    item["rdkit_formula"] for item in descriptions.loc[valid]
]

library = library.reset_index(drop=True)

print("Unique training structures:", train["inchikey14"].nunique())
print("Valid candidate structures:", len(library))
print("Missing exact masses:", library["exact_mass"].isna().sum())
display(library.head())

In [ ]:
import pandas as pd

DATA_DIR = "/kaggle/input/datasets/denisketer/enveda-casmi-2026"

train_meta = pd.read_parquet(
    f"{DATA_DIR}/train.parquet",
    columns=[
        "inchikey14",
        "precursor_mz",
        "adduct",
        "ionization_mode",
        "precursor_error_ppm",
    ],
)

print("Rows:", len(train_meta))
print("\nMost common adducts:")
display(train_meta["adduct"].value_counts().head(20).to_frame("count"))

print("\nIonization modes:")
display(train_meta["ionization_mode"].value_counts(dropna=False).to_frame("count"))

print("\nPrecursor error (ppm) summary:")
display(train_meta["precursor_error_ppm"].describe(
    percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]
).to_frame("precursor_error_ppm"))

In [ ]:
test_meta = pd.read_parquet(
    f"{DATA_DIR}/test.parquet",
    columns=[
        "molecule_id",
        "spectrum_id",
        "precursor_mz",
        "adduct",
        "ionization_mode",
    ],
)

print("Test spectra:", len(test_meta))
print("Test molecules:", test_meta["molecule_id"].nunique())
print("\nTest adducts:")
display(test_meta["adduct"].value_counts(dropna=False).to_frame("count"))

print("\nExample test spectra:")
display(test_meta.head(10))

In [ ]:
import pandas as pd

DATA_DIR = "/kaggle/input/datasets/denisketer/enveda-casmi-2026"

test_meta = pd.read_parquet(
    f"{DATA_DIR}/test.parquet",
    columns=[
        "molecule_id",
        "spectrum_id",
        "precursor_mz",
        "adduct",
        "ionization_mode",
    ],
)

print("Test spectra:", len(test_meta))
print("Test molecules:", test_meta["molecule_id"].nunique())

print("\nTest adducts:")
display(test_meta["adduct"].value_counts(dropna=False).to_frame("count"))

print("\nIonization modes:")
display(test_meta["ionization_mode"].value_counts(dropna=False).to_frame("count"))

print("\nExample spectra:")
display(test_meta.head(10))


In [ ]:
import numpy as np
import pandas as pd
from collections import defaultdict

# Exact mass offsets for the adducts observed in the test set.
# The offsets convert neutral molecular exact mass to ion m/z.
PROTON = 1.0072764666

ADDUCT_OFFSETS = {
    "[M+H]+":       PROTON,
    "[M-H]-":      -PROTON,
    "[M+CH2O2-H]-": 44.998201,
    "[M+Na]+":      22.989218,
    "[M+NH4]+":     18.033823,
    "[M+K]+":       38.963158,
    "[M+Cl]-":      34.969402,
}

# Prepare the candidate mass catalog for each supported adduct.
candidate_catalog = {}

for adduct, offset in ADDUCT_OFFSETS.items():
    masses = library["exact_mass"].to_numpy(dtype=np.float64) + offset
    order = np.argsort(masses)
    candidate_catalog[adduct] = (
        masses[order],
        library["inchikey14"].to_numpy()[order],
        library["normalized_smiles"].to_numpy()[order],
    )

# Build a validation sample from ordinary precursor measurements.
# This is a retrieval sanity check, not a fully leakage-proof estimate
# of performance on unseen chemical structures.
val = train_meta[
    train_meta["adduct"].isin(ADDUCT_OFFSETS)
    & train_meta["precursor_mz"].between(50, 2000)
    & train_meta["precursor_mz"].notna()
    & train_meta["inchikey14"].notna()
].sample(n=min(5000, len(train_meta[
    train_meta["adduct"].isin(ADDUCT_OFFSETS)
    & train_meta["precursor_mz"].between(50, 2000)
    & train_meta["precursor_mz"].notna()
    & train_meta["inchikey14"].notna()
])), random_state=42)

TOLERANCE_PPM = 20
reciprocal_ranks = []
covered = 0
evaluated = 0

for row in val.itertuples(index=False):
    adduct = row.adduct
    observed_mz = float(row.precursor_mz)
    target_key = row.inchikey14

    masses, keys, smiles = candidate_catalog[adduct]
    tolerance = observed_mz * TOLERANCE_PPM / 1e6

    left = np.searchsorted(masses, observed_mz - tolerance, side="left")
    right = np.searchsorted(masses, observed_mz + tolerance, side="right")

    if left == right:
        reciprocal_ranks.append(0.0)
        evaluated += 1
        continue

    local_masses = masses[left:right]
    local_keys = keys[left:right]

    errors_ppm = np.abs(local_masses - observed_mz) / observed_mz * 1e6
    order = np.argsort(errors_ppm, kind="stable")
    ranked_keys = local_keys[order][:25]

    matches = np.flatnonzero(ranked_keys == target_key)
    if len(matches):
        rank = int(matches[0]) + 1
        reciprocal_ranks.append(1.0 / rank)
        covered += 1
    else:
        reciprocal_ranks.append(0.0)

    evaluated += 1

print(f"Validation spectra evaluated: {evaluated:,}")
print(f"Target found in top 25: {covered:,} ({covered / evaluated:.1%})")
print(f"Mass-retrieval MRR@25: {np.mean(reciprocal_ranks):.5f}")
print(f"Mass tolerance: ±{TOLERANCE_PPM} ppm")
print(f"Candidate structures: {len(library):,}")

In [ ]:
# Uses the existing train DataFrame if it is still in memory.
# Otherwise, read only the necessary columns.

import pandas as pd

DATA_DIR = "/kaggle/input/datasets/denisketer/enveda-casmi-2026"

train_check = pd.read_parquet(
    f"{DATA_DIR}/train.parquet",
    columns=["inchikey14", "normalized_smiles"],
)

smiles_per_key = (
    train_check.groupby("inchikey14")["normalized_smiles"]
    .nunique()
)

print("Unique InChIKey14:", train_check["inchikey14"].nunique())
print("Keys with multiple SMILES:", (smiles_per_key > 1).sum())
print("Maximum SMILES per key:", smiles_per_key.max())

spectra_per_key = train_check.groupby("inchikey14").size()

print("\nSpectra per structure:")
print(spectra_per_key.describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.99]
))

In [ ]:

import numpy as np
import pandas as pd

DATA_DIR = "/kaggle/input/datasets/denisketer/enveda-casmi-2026"

# Use one representative SMILES per InChIKey14.
structures = (
    train_check[["inchikey14", "normalized_smiles"]]
    .dropna()
    .drop_duplicates(["inchikey14", "normalized_smiles"])
    .sort_values(["inchikey14", "normalized_smiles"])
    .drop_duplicates("inchikey14")
    .reset_index(drop=True)
)

# Deterministic split by structure identity.
rng = np.random.default_rng(42)
indices = rng.permutation(len(structures))
n_val = int(round(0.10 * len(structures)))

val_keys = set(structures.iloc[indices[:n_val]]["inchikey14"])
train_keys = set(structures.iloc[indices[n_val:]]["inchikey14"])

assert val_keys.isdisjoint(train_keys)

print("Total structures:", len(structures))
print("Training structures:", len(train_keys))
print("Validation structures:", len(val_keys))
print("Overlap:", len(train_keys & val_keys))

# All spectra belonging to validation structures are held out.
train_spectra = train_meta[train_meta["inchikey14"].isin(train_keys)].copy()
val_spectra = train_meta[train_meta["inchikey14"].isin(val_keys)].copy()

print("\nTraining spectra:", len(train_spectra))
print("Validation spectra:", len(val_spectra))
print("Validation structures represented:", val_spectra["inchikey14"].nunique())


In [ ]:

import pandas as pd
import numpy as np

DATA_DIR = "/kaggle/input/datasets/denisketer/enveda-casmi-2026"

spectra = pd.read_parquet(
    f"{DATA_DIR}/train.parquet",
    columns=[
        "inchikey14",
        "normalized_smiles",
        "precursor_mz",
        "adduct",
        "ionization_mode",
        "ms2_mzs",
        "ms2_normalized_intensities",
    ],
)

rng = np.random.default_rng(42)

# Shuffle rows within each structure deterministically.
spectra["_random"] = rng.random(len(spectra))
spectra = spectra.sort_values(
    ["inchikey14", "_random"]
).reset_index(drop=True)

# Reserve approximately 20% of each structure's spectra,
# but never remove the final reference spectrum.
group_sizes = spectra.groupby("inchikey14")["inchikey14"].transform("size")
group_position = spectra.groupby("inchikey14").cumcount()
n_holdout = np.floor(group_sizes * 0.20).astype(int)
n_holdout = np.minimum(n_holdout, group_sizes - 1)

is_validation = group_position < n_holdout

validation_spectra = spectra.loc[is_validation].drop(
    columns="_random"
).copy()

reference_spectra = spectra.loc[~is_validation].drop(
    columns="_random"
).copy()

assert set(validation_spectra["inchikey14"]).issubset(
    set(reference_spectra["inchikey14"])
)

print("Reference spectra:", len(reference_spectra))
print("Held-out spectra:", len(validation_spectra))
print("Structures in reference:", reference_spectra["inchikey14"].nunique())
print("Structures in validation:", validation_spectra["inchikey14"].nunique())
print("Structure overlap:", len(
    set(reference_spectra["inchikey14"])
    & set(validation_spectra["inchikey14"])
))


In [ ]:
print("Reference columns:", reference_spectra.columns.tolist())

print("\nExample fragment m/z values:")
print(reference_spectra["ms2_mzs"].iloc[0][:10])

print("\nExample normalized intensities:")
print(reference_spectra["ms2_normalized_intensities"].iloc[0][:10])

print("\nReference adduct counts:")
display(reference_spectra["adduct"].value_counts().head(10).to_frame("count"))

print("\nValidation adduct counts:")
display(validation_spectra["adduct"].value_counts().head(10).to_frame("count"))

In [ ]:

import pandas as pd
import numpy as np

DATA_DIR = "/kaggle/input/datasets/denisketer/enveda-casmi-2026"

spectra = pd.read_parquet(
    f"{DATA_DIR}/train.parquet",
    columns=[
        "inchikey14",
        "normalized_smiles",
        "precursor_mz",
        "adduct",
        "ionization_mode",
        "ms2_mzs",
        "ms2_normalized_intensities",
    ],
)

rng = np.random.default_rng(42)
spectra["_random"] = rng.random(len(spectra))
spectra = spectra.sort_values(
    ["inchikey14", "_random"]
).reset_index(drop=True)

group_sizes = spectra.groupby("inchikey14")["inchikey14"].transform("size")
group_position = spectra.groupby("inchikey14").cumcount()
n_holdout = np.floor(group_sizes * 0.20).astype(int)
n_holdout = np.minimum(n_holdout, group_sizes - 1)

is_validation = group_position < n_holdout

validation_spectra = (
    spectra.loc[is_validation]
    .drop(columns="_random")
    .copy()
)

reference_spectra = (
    spectra.loc[~is_validation]
    .drop(columns="_random")
    .copy()
)

print("Reference spectra:", len(reference_spectra))
print("Validation spectra:", len(validation_spectra))
print(
    "Structures represented in both:",
    len(
        set(reference_spectra["inchikey14"])
        & set(validation_spectra["inchikey14"])
    ),
)



In [ ]:

import numpy as np
import pandas as pd
from collections import defaultdict
from sklearn.feature_extraction import DictVectorizer
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity

# Keep the first experiment small and reproducible.
N_QUERY = 2000
FRAGMENT_BIN_WIDTH = 0.02
MIN_FRAGMENT_MZ = 50.0
MAX_FRAGMENT_MZ = 1000.0

rng = np.random.default_rng(42)
query_idx = rng.choice(
    len(validation_spectra),
    size=min(N_QUERY, len(validation_spectra)),
    replace=False,
)
queries = validation_spectra.iloc[query_idx].reset_index(drop=True)

# Convert each spectrum into a sparse dictionary of binned peak intensities.
def spectrum_to_dict(mzs, intensities):
    features = {}
    for mz, intensity in zip(mzs, intensities):
        mz = float(mz)
        intensity = float(intensity)

        if not np.isfinite(mz) or not np.isfinite(intensity):
            continue
        if mz < MIN_FRAGMENT_MZ or mz > MAX_FRAGMENT_MZ or intensity <= 0:
            continue

        bin_id = int(round(mz / FRAGMENT_BIN_WIDTH))
        features[bin_id] = features.get(bin_id, 0.0) + intensity

    # Square-root weighting reduces domination by the base peak.
    return {k: np.sqrt(v) for k, v in features.items()}

print("Query spectra:", len(queries))
print("Reference spectra:", len(reference_spectra))
print("Fragment bin width:", FRAGMENT_BIN_WIDTH)
print("Fragment m/z range:", MIN_FRAGMENT_MZ, "to", MAX_FRAGMENT_MZ)


In [ ]:

# Sample background reference spectra.
N_BACKGROUND = 20_000

background = reference_spectra.sample(
    n=min(N_BACKGROUND, len(reference_spectra)),
    random_state=42,
)

# Add one reference spectrum for each query target, where available.
query_keys = set(queries["inchikey14"])

target_refs = (
    reference_spectra[
        reference_spectra["inchikey14"].isin(query_keys)
    ]
    .groupby("inchikey14", sort=False)
    .head(1)
)

# Deduplicate using scalar columns only; fragment arrays are unhashable.
refs = pd.concat(
    [background, target_refs],
    axis=0,
).drop_duplicates(
    subset=[
        "inchikey14",
        "precursor_mz",
        "adduct",
        "ionization_mode",
    ],
).reset_index(drop=True)

print("Queries:", len(queries))
print("Reference spectra sampled:", len(refs))
print(
    "Query target structures with references:",
    len(query_keys & set(refs["inchikey14"])),
)



In [ ]:

import numpy as np
from sklearn.feature_extraction import DictVectorizer
from sklearn.preprocessing import normalize

def spectrum_to_dict(mzs, intensities):
    features = {}
    for mz, intensity in zip(mzs, intensities):
        mz, intensity = float(mz), float(intensity)
        if not np.isfinite(mz) or not np.isfinite(intensity):
            continue
        if 50 <= mz <= 1000 and intensity > 0:
            bin_id = int(round(mz / 0.02))
            features[bin_id] = features.get(bin_id, 0.0) + intensity
    return {k: np.sqrt(v) for k, v in features.items()}

ref_dicts = [
    spectrum_to_dict(mzs, ints)
    for mzs, ints in zip(
        refs["ms2_mzs"], refs["ms2_normalized_intensities"]
    )
]
query_dicts = [
    spectrum_to_dict(mzs, ints)
    for mzs, ints in zip(
        queries["ms2_mzs"], queries["ms2_normalized_intensities"]
    )
]

vectorizer = DictVectorizer(dtype=np.float32)
R = normalize(vectorizer.fit_transform(ref_dicts), norm="l2")
Q = normalize(vectorizer.transform(query_dicts), norm="l2")

ref_modes = refs["ionization_mode"].fillna("unknown").to_numpy()
ref_keys = refs["inchikey14"].to_numpy()
query_modes = queries["ionization_mode"].fillna("unknown").to_numpy()
query_keys_arr = queries["inchikey14"].to_numpy()

rr = []
top25_hits = 0

for i in range(len(queries)):
    compatible = np.flatnonzero(ref_modes == query_modes[i])
    if len(compatible) == 0:
        rr.append(0.0)
        continue

    sims = (Q[i] @ R[compatible].T).toarray().ravel()

    best_by_key = {}
    for j, score in zip(compatible, sims):
        key = ref_keys[j]
        if score > best_by_key.get(key, -1):
            best_by_key[key] = float(score)

    ranked_keys = [
        key for key, score in sorted(
            best_by_key.items(), key=lambda item: item[1], reverse=True
        )[:25]
    ]

    if query_keys_arr[i] in ranked_keys:
        rank = ranked_keys.index(query_keys_arr[i]) + 1
        rr.append(1.0 / rank)
        top25_hits += 1
    else:
        rr.append(0.0)

print("Queries evaluated:", len(queries))
print("Target in top 25:", f"{top25_hits / len(queries):.1%}")
print("MRR@25:", f"{np.mean(rr):.5f}")



In [ ]:

import numpy as np

ref_modes = refs["ionization_mode"].fillna("unknown").to_numpy()
ref_keys = refs["inchikey14"].to_numpy()
query_modes = queries["ionization_mode"].fillna("unknown").to_numpy()
query_keys_arr = queries["inchikey14"].to_numpy()

rr_fair = []
hits_fair = 0

for i in range(len(queries)):
    # Same ionization mode, but exclude every reference spectrum
    # belonging to the query's true structure.
    compatible = np.flatnonzero(
        (ref_modes == query_modes[i])
        & (ref_keys != query_keys_arr[i])
    )

    # This exclusion is intentionally strict: the correct structure
    # cannot be retrieved from this reference set.
    if len(compatible) == 0:
        rr_fair.append(0.0)
        continue

    sims = (Q[i] @ R[compatible].T).toarray().ravel()

    best_by_key = {}
    for j, score in zip(compatible, sims):
        key = ref_keys[j]
        if score > best_by_key.get(key, -1):
            best_by_key[key] = float(score)

    ranked_keys = [
        key for key, score in sorted(
            best_by_key.items(),
            key=lambda item: item[1],
            reverse=True,
        )[:25]
    ]

    # Since the true structure is excluded, its rank is necessarily
    # outside this reference catalog. Keep this diagnostic explicit.
    rr_fair.append(0.0)

print("Queries evaluated:", len(queries))
print("Target-in-reference candidates:", 0)
print("MRR@25:", f"{np.mean(rr_fair):.5f}")


In [ ]:
# 1. Confirm query labels are present in the reference catalog.
ref_label_set = set(refs_all["inchikey14"].astype(str))
query_labels = queries["inchikey14"].astype(str)

print("Queries:", len(queries))
print("Query labels found in reference catalog:",
      query_labels.isin(ref_label_set).sum())

# 2. Check whether the query and reference labels have matching formats.
print("\nQuery label examples:")
print(query_labels.head().tolist())

print("\nReference label examples:")
print(refs_all["inchikey14"].astype(str).head().tolist())

# 3. Check ionization-mode compatibility for the true structures.
ref_mode_by_label = (
    refs_all.assign(
        inchikey14=refs_all["inchikey14"].astype(str),
        ionization_mode=refs_all["ionization_mode"].astype(str),
    )
    .groupby("inchikey14")["ionization_mode"]
    .agg(set)
    .to_dict()
)

mode_matches = sum(
    str(mode) in ref_mode_by_label.get(str(label), set())
    for label, mode in zip(
        queries["inchikey14"], queries["ionization_mode"]
    )
)

print("\nTargets with a reference in the same ionization mode:", mode_matches)

In [ ]:
# Inspect the first 10 queries against their true reference structures.
for i in range(10):
    true_label = str(queries.iloc[i]["inchikey14"])
    query_mode = str(queries.iloc[i]["ionization_mode"])

    # Reference rows for this exact structure and ionization mode
    mask = (
        (refs_all["inchikey14"].astype(str) == true_label)
        & (refs_all["ionization_mode"].astype(str) == query_mode)
    )
    ref_idx = np.flatnonzero(mask.to_numpy())

    if len(ref_idx) == 0:
        print(i, "No matching-mode reference")
        continue

    # Similarity between this query and its true reference spectrum
    sim = float((Q[i] @ R[ref_idx].T).toarray().max())

    print(
        i,
        "true label:", true_label,
        "| mode:", query_mode,
        "| query features:", Q[i].nnz,
        "| reference features:", R[ref_idx].nnz,
        "| true similarity:", round(sim, 6),
    )

In [ ]:
# Recompute ranks for a small sample, explicitly checking sparse indices.
for i in range(10):
    true_label = str(queries.iloc[i]["inchikey14"])
    query_mode = str(queries.iloc[i]["ionization_mode"])

    compatible_idx = np.flatnonzero(ref_modes == query_mode)
    sims = (Q[i] @ R[compatible_idx].T).tocsr()

    # Candidate positions returned by the sparse multiplication
    positions = sims.indices
    scores = sims.data
    labels = ref_labels[compatible_idx[positions]]

    if len(scores) == 0:
        print(i, "No positive similarities")
        continue

    order = np.argsort(-scores, kind="stable")
    labels_ranked = labels[order]
    scores_ranked = scores[order]

    hits = np.flatnonzero(labels_ranked == true_label)

    if len(hits):
        rank = int(hits[0]) + 1
        print(
            i,
            "true similarity found:",
            round(float(scores_ranked[hits[0]]), 6),
            "| rank:", rank,
        )
    else:
        print(
            i,
            "target not among positive-score candidates",
            "| positive candidates:", len(scores),
        )

In [ ]:
reciprocal_ranks = []
hits_at_25 = []
rank_counts = {}

for start in range(0, len(queries), 32):
    end = min(start + 32, len(queries))
    similarities = Q[start:end] @ R.T

    for local_i, global_i in enumerate(range(start, end)):
        true_label = str(queries.iloc[global_i]["inchikey14"])
        query_mode = str(queries.iloc[global_i]["ionization_mode"])

        # Keep only candidates in the same ionization mode.
        compatible_idx = np.flatnonzero(ref_modes == query_mode)
        row = (Q[global_i] @ R[compatible_idx].T).tocsr()

        positions = row.indices
        scores = row.data
        labels = ref_labels[compatible_idx[positions]]

        order = np.argsort(-scores, kind="stable")
        ranked_labels = labels[order]
        matches = np.flatnonzero(ranked_labels == true_label)

        if len(matches) and matches[0] < 25:
            rank = int(matches[0]) + 1
            reciprocal_ranks.append(1.0 / rank)
            hits_at_25.append(1)
            rank_counts[rank] = rank_counts.get(rank, 0) + 1
        else:
            reciprocal_ranks.append(0.0)
            hits_at_25.append(0)

print("Queries evaluated:", len(queries))
print(f"MRR@25: {np.mean(reciprocal_ranks):.5f}")
print(f"Target in top 25: {np.mean(hits_at_25):.1%}")
print("Exact rank-1 hits:", rank_counts.get(1, 0))

In [ ]:
reciprocal_ranks = []
hits_at_25 = []
rank_counts = {}

for start in range(0, len(queries), 32):
    end = min(start + 32, len(queries))
    similarities = Q[start:end] @ R.T

    for local_i, global_i in enumerate(range(start, end)):
        true_label = str(queries.iloc[global_i]["inchikey14"])
        query_mode = str(queries.iloc[global_i]["ionization_mode"])

        # Keep only candidates in the same ionization mode.
        compatible_idx = np.flatnonzero(ref_modes == query_mode)
        row = (Q[global_i] @ R[compatible_idx].T).tocsr()

        positions = row.indices
        scores = row.data
        labels = ref_labels[compatible_idx[positions]]

        order = np.argsort(-scores, kind="stable")
        ranked_labels = labels[order]
        matches = np.flatnonzero(ranked_labels == true_label)

        if len(matches) and matches[0] < 25:
            rank = int(matches[0]) + 1
            reciprocal_ranks.append(1.0 / rank)
            hits_at_25.append(1)
            rank_counts[rank] = rank_counts.get(rank, 0) + 1
        else:
            reciprocal_ranks.append(0.0)
            hits_at_25.append(0)

print("Queries evaluated:", len(queries))
print(f"MRR@25: {np.mean(reciprocal_ranks):.5f}")
print(f"Target in top 25: {np.mean(hits_at_25):.1%}")
print("Exact rank-1 hits:", rank_counts.get(1, 0))

In [ ]:
# Rebuild one reference spectrum per structure
refs_all = (
    reference_spectra
    .groupby("inchikey14", sort=False)
    .head(1)
    .reset_index(drop=True)
)

print("Reference spectra:", len(refs_all))
print("Reference structures:", refs_all["inchikey14"].nunique())

In [ ]:
required = ["queries", "refs_all", "Q", "R", "np"]

for name in required:
    print(f"{name}: {'OK' if name in globals() else 'MISSING'}")

In [ ]:
print("refs_all rows:", len(refs_all))
print("R shape:", R.shape)
print("Q shape:", Q.shape)
print("Reference labels:", len(ref_labels))

In [ ]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.preprocessing import normalize
import numpy as np

# Recreate the reference set
refs_all = (
    reference_spectra
    .groupby("inchikey14", sort=False)
    .head(1)
    .reset_index(drop=True)
)

# Recreate features in exactly this order
ref_dicts = [
    spectrum_to_dict(mz, intensity)
    for mz, intensity in zip(
        refs_all["ms2_mzs"],
        refs_all["ms2_normalized_intensities"],
    )
]

query_dicts = [
    spectrum_to_dict(mz, intensity)
    for mz, intensity in zip(
        queries["ms2_mzs"],
        queries["ms2_normalized_intensities"],
    )
]

vectorizer = DictVectorizer(dtype=np.float32)
R = normalize(vectorizer.fit_transform(ref_dicts), norm="l2")
Q = normalize(vectorizer.transform(query_dicts), norm="l2")

ref_labels = refs_all["inchikey14"].astype(str).to_numpy()
ref_modes = refs_all["ionization_mode"].astype(str).to_numpy()

print("Reference rows:", len(refs_all))
print("R rows:", R.shape[0])
print("Query rows:", Q.shape[0])

In [ ]:
print("reference_spectra:", len(reference_spectra))
print("refs_all:", len(refs_all))
print("R shape:", R.shape)
print("Q shape:", Q.shape)
print("queries:", len(queries))

In [ ]:
import numpy as np
from sklearn.feature_extraction import DictVectorizer
from sklearn.preprocessing import normalize

# Rebuild reference spectra and feature matrix as one consistent unit
refs_all = (
    reference_spectra
    .groupby("inchikey14", sort=False)
    .head(1)
    .reset_index(drop=True)
)

ref_dicts = [
    spectrum_to_dict(mz, intensity)
    for mz, intensity in zip(
        refs_all["ms2_mzs"],
        refs_all["ms2_normalized_intensities"],
    )
]

R = normalize(
    vectorizer.transform(ref_dicts),
    norm="l2",
)

ref_labels = refs_all["inchikey14"].astype(str).to_numpy()
ref_modes = refs_all["ionization_mode"].astype(str).to_numpy()

assert R.shape[0] == len(refs_all) == len(ref_labels) == len(ref_modes)

print("Reference rows:", len(refs_all))
print("Matrix shape:", R.shape)
print("Reference labels:", len(ref_labels))

In [ ]:
i = 4
true_label = str(queries.iloc[i]["inchikey14"])
query_mode = str(queries.iloc[i]["ionization_mode"])

mask = (
    (ref_labels == true_label)
    & (ref_modes == query_mode)
)
ref_idx = np.flatnonzero(mask)

if len(ref_idx):
    sim = float((Q[i] @ R[ref_idx].T).toarray().max())
    print("True-structure similarity:", round(sim, 6))
else:
    print("No reference for this structure in the same ionization mode.")

In [ ]:
i = 4

true_label = str(queries.iloc[i]["inchikey14"])
query_mode = str(queries.iloc[i]["ionization_mode"])

mask = (
    (ref_labels == true_label)
    & (ref_modes == query_mode)
)
ref_idx = np.flatnonzero(mask)

if len(ref_idx):
    sim = float((Q[i] @ R[ref_idx].T).toarray().max())
    print("True-structure similarity:", round(sim, 6))
else:
    print("No matching-mode reference found.")

In [ ]:
# Step 3A: Verify the current benchmark state

print("Queries:", len(queries))
print("Reference structures:", len(refs_all))
print("Query feature matrix:", Q.shape)
print("Reference feature matrix:", R.shape)

assert Q.shape[0] == len(queries)
assert R.shape[0] == len(refs_all)
assert R.shape[0] == len(ref_labels)
assert R.shape[0] == len(ref_modes)

print("\nPASS: Matrix dimensions and reference labels are aligned.")

In [ ]:
# Step 3B: Inspect the adducts we need to support

print("Test-set adduct counts:")
print(test["adduct"].value_counts(dropna=False))

print("\nTraining-set adduct counts:")
print(train["adduct"].value_counts().head(12))

In [ ]:
print("Training variable type:", type(train))
print("\nAvailable columns:")
print(train.columns.tolist())

In [ ]:
for name, obj in list(globals().items()):
    if isinstance(obj, __import__("pandas").DataFrame):
        print(name, obj.shape, list(obj.columns[:8]))

In [ ]:
print("Training adduct counts:")
print(train_meta["adduct"].value_counts().head(12))

print("\nTest adduct counts:")
print(test["adduct"].value_counts())

print("\nTraining ionization modes:")
print(train_meta["ionization_mode"].value_counts())

In [ ]:
# Step 3D: Validate adduct offsets against training records

import numpy as np
import pandas as pd

adduct_offsets = {
    "[M+H]+": 1.007276,
    "[M-H]-": -1.007276,
    "[M+Na]+": 22.989218,
    "[M+NH4]+": 18.033823,
    "[M+K]+": 38.963158,
    "[M+Cl]-": 34.969402,
    "[M+CH2O2-H]-": 44.998201,
}

# Join a sample of training records to their RDKit exact masses.
mass_check = (
    train_meta[["inchikey14", "precursor_mz", "adduct"]]
    .merge(
        library[["inchikey14", "exact_mass"]],
        on="inchikey14",
        how="inner",
    )
)

mass_check = mass_check[
    mass_check["adduct"].isin(adduct_offsets)
].copy()

mass_check["offset"] = mass_check["adduct"].map(adduct_offsets)
mass_check["predicted_mz"] = (
    mass_check["exact_mass"] + mass_check["offset"]
)
mass_check["error_ppm_calc"] = (
    (mass_check["precursor_mz"] - mass_check["predicted_mz"])
    / mass_check["predicted_mz"]
    * 1e6
)

print(
    mass_check.groupby("adduct")["error_ppm_calc"]
    .agg(["count", "median", "mean"])
    .round(3)
)

In [ ]:
error_summary = (
    mass_check.groupby("adduct")["error_ppm_calc"]
    .quantile([0.50, 0.90, 0.95, 0.99])
    .unstack()
)
error_summary.columns = ["p50", "p90", "p95", "p99"]

error_summary["within_10_ppm"] = (
    mass_check.assign(
        within_10=mass_check["error_ppm_calc"].abs() <= 10
    )
    .groupby("adduct")["within_10"]
    .mean() * 100
)

print(error_summary.round(2))

In [ ]:
# Count how many validation spectra have a matching structure
# within ±10 ppm, using the expected adduct mass offset.

val_mass = val_spectra[
    ["inchikey14", "precursor_mz", "adduct"]
].merge(
    library[["inchikey14", "exact_mass"]],
    on="inchikey14",
    how="left"
)

val_mass["offset"] = val_mass["adduct"].map(adduct_offsets)
val_mass["predicted_mz"] = val_mass["exact_mass"] + val_mass["offset"]

val_mass["error_ppm"] = (
    (val_mass["precursor_mz"] - val_mass["predicted_mz"])
    / val_mass["predicted_mz"] * 1e6
)

val_mass["within_10_ppm"] = val_mass["error_ppm"].abs() <= 10

print("Validation spectra:", len(val_mass))
print(
    "Known structure within ±10 ppm:",
    f"{val_mass['within_10_ppm'].mean() * 100:.2f}%"
)
print("\nBy adduct:")
print(
    val_mass.groupby("adduct")["within_10_ppm"]
    .agg(["count", "mean"])
    .assign(percent=lambda x: x["mean"] * 100)
    .drop(columns="mean")
    .round(2)
)

In [ ]:
unsupported = (
    val_spectra.loc[
        ~val_spectra["adduct"].isin(adduct_offsets),
        "adduct"
    ]
    .value_counts()
)

print("Validation spectra with unsupported adducts:", unsupported.sum())
print("\nMost common unsupported adducts:")
print(unsupported.head(20))

In [ ]:
import re

PROTON = 1.0072764666
SODIUM = 22.989218
POTASSIUM = 38.963158
AMMONIUM = 18.033823
CHLORIDE = 34.969402
FORMIC_ACID_MINUS_H = 44.998201
WATER = 18.010565

def expected_mz(neutral_mass, adduct):
    # Dimer ions
    if adduct == "[2M+H]+":
        return (2 * neutral_mass + PROTON)
    if adduct == "[2M+Na]+":
        return (2 * neutral_mass + SODIUM)
    if adduct == "[2M+NH4]+":
        return (2 * neutral_mass + AMMONIUM)
    if adduct == "[2M-H]-":
        return (2 * neutral_mass - PROTON)
    if adduct == "[2M+CH2O2-H]-":
        return (2 * neutral_mass + FORMIC_ACID_MINUS_H)

    # Common singly charged ions
    offsets = {
        "[M+H]+": PROTON,
        "[M-H]-": -PROTON,
        "[M+Na]+": SODIUM,
        "[M+K]+": POTASSIUM,
        "[M+NH4]+": AMMONIUM,
        "[M+Cl]-": CHLORIDE,
        "[M+CH2O2-H]-": FORMIC_ACID_MINUS_H,
        "[M]+": 0.0,
        "[M]-": 0.0,
        "[M-H2O+H]+": PROTON - WATER,
        "[M-2H2O+H]+": PROTON - 2 * WATER,
        "[M-H2O]+": -WATER,
        "[M-H2O-H]-": -WATER - PROTON,
        "[M-2H]-": -2 * PROTON,
        "[M-CH3]-": -15.023475,
    }
    if adduct in offsets:
        return neutral_mass + offsets[adduct]

    # Multiply charged ions
    if adduct == "[M+2H]2+":
        return (neutral_mass + 2 * PROTON) / 2
    if adduct == "[M+3H]3+":
        return (neutral_mass + 3 * PROTON) / 3

    return None

print("Calculator defined. Next, we'll test its coverage on validation spectra.")

In [ ]:
# Calculate expected precursor m/z for each validation spectrum.
val_mass = val_spectra[
    ["inchikey14", "precursor_mz", "adduct"]
].merge(
    library[["inchikey14", "exact_mass"]],
    on="inchikey14",
    how="left"
)

val_mass["predicted_mz"] = [
    expected_mz(mass, adduct)
    for mass, adduct in zip(
        val_mass["exact_mass"],
        val_mass["adduct"]
    )
]

val_mass["error_ppm"] = (
    (val_mass["precursor_mz"] - val_mass["predicted_mz"])
    / val_mass["predicted_mz"] * 1e6
)

supported = (
    val_mass["predicted_mz"].notna()
    & np.isfinite(val_mass["predicted_mz"])
    & (val_mass["predicted_mz"] > 0)
)

print(f"Validation spectra: {len(val_mass):,}")
print(f"Supported adducts: {supported.sum():,} ({supported.mean()*100:.2f}%)")
print(f"Unsupported adducts: {(~supported).sum():,}")

print("\nError percentiles for supported adducts:")
print(
    val_mass.loc[supported, "error_ppm"]
    .abs()
    .quantile([0.50, 0.90, 0.95, 0.99])
    .rename(index={
        0.50: "median absolute ppm",
        0.90: "90th percentile",
        0.95: "95th percentile",
        0.99: "99th percentile",
    })
    .round(2)
)

print("\nFraction within tolerance:")
for ppm in [5, 10, 20, 50]:
    fraction = (
        val_mass.loc[supported, "error_ppm"].abs() <= ppm
    ).mean() * 100
    print(f"±{ppm:>2} ppm: {fraction:.2f}%")

In [ ]:
# Summarize mass errors by adduct for supported validation spectra.
val_mass["abs_error_ppm"] = val_mass["error_ppm"].abs()

supported_rows = val_mass[
    val_mass["predicted_mz"].notna()
    & np.isfinite(val_mass["error_ppm"])
    & (val_mass["predicted_mz"] > 0)
].copy()

adduct_error_summary = (
    supported_rows.groupby("adduct")
    .agg(
        count=("error_ppm", "size"),
        median_abs_ppm=("abs_error_ppm", "median"),
        p95_abs_ppm=("abs_error_ppm", lambda x: x.quantile(0.95)),
        pct_within_10ppm=(
            "abs_error_ppm",
            lambda x: (x <= 10).mean() * 100
        ),
        pct_over_100ppm=(
            "abs_error_ppm",
            lambda x: (x > 100).mean() * 100
        ),
    )
    .sort_values("count", ascending=False)
)

print(adduct_error_summary.round(2).to_string())

In [ ]:
# Summarize the error distribution by structure and adduct.
outlier_summary = (
    supported_rows
    .groupby(["inchikey14", "adduct"])
    .agg(
        rows=("error_ppm", "size"),
        median_abs_ppm=("abs_error_ppm", "median"),
        max_abs_ppm=("abs_error_ppm", "max"),
        observed_mz_min=("precursor_mz", "min"),
        observed_mz_max=("precursor_mz", "max"),
        predicted_mz=("predicted_mz", "median"),
    )
    .reset_index()
)

outlier_summary = outlier_summary[
    outlier_summary["max_abs_ppm"] > 100
].sort_values("max_abs_ppm", ascending=False)

print("Structure/adduct groups with errors >100 ppm:",
      len(outlier_summary))

print("\nLargest 20 groups:")
print(outlier_summary.head(20).to_string(index=False))

In [ ]:
# Inspect the largest mass errors without changing any data.
outlier_cols = [
    "inchikey14",
    "adduct",
    "precursor_mz",
    "exact_mass",
    "predicted_mz",
    "error_ppm",
]

worst_errors = (
    supported_rows.loc[
        supported_rows["abs_error_ppm"] > 100,
        outlier_cols
    ]
    .assign(
        abs_error_ppm=lambda x: x["error_ppm"].abs()
    )
    .sort_values("abs_error_ppm", ascending=False)
)

print("Rows with absolute error >100 ppm:", len(worst_errors))
print("\nCounts by adduct:")
print(worst_errors["adduct"].value_counts().head(15))

print("\nLargest 15 errors:")
print(worst_errors.head(15).to_string(index=False))

In [ ]:
# Inspect individual validation rows for one suspicious structure.
example_key = "FASDKYOPVNHBLU"

example_rows = (
    supported_rows.loc[
        supported_rows["inchikey14"] == example_key,
        [
            "inchikey14",
            "adduct",
            "precursor_mz",
            "exact_mass",
            "predicted_mz",
            "error_ppm",
        ],
    ]
    .copy()
)

example_rows["abs_error_ppm"] = example_rows["error_ppm"].abs()
example_rows = example_rows.sort_values("abs_error_ppm", ascending=False)

print("Rows for structure:", len(example_rows))
print("\nLargest errors:")
print(example_rows.head(10).to_string(index=False))

print("\nSmallest errors:")
print(
    example_rows.sort_values("abs_error_ppm")
    .head(10)
    .to_string(index=False)
)

In [ ]:
# Categorize mass errors for each supported adduct.
supported_rows["error_category"] = pd.cut(
    supported_rows["abs_error_ppm"],
    bins=[-1, 10, 50, 100, float("inf")],
    labels=["<=10 ppm", "10-50 ppm", "50-100 ppm", ">100 ppm"]
)

error_counts = pd.crosstab(
    supported_rows["adduct"],
    supported_rows["error_category"],
    normalize="index"
) * 100

# Show the most frequent adducts first.
adduct_order = supported_rows["adduct"].value_counts().index
error_counts = error_counts.reindex(adduct_order)

print(error_counts.head(15).round(2).to_string())

In [ ]:
# Evaluate whether the known structure passes a mass filter.
# This tests mass coverage, not final ranking accuracy.

rng = np.random.default_rng(42)

mass_test = val_spectra.sample(
    n=min(500, len(val_spectra)),
    random_state=42
).copy()

mass_test = mass_test.merge(
    library[["inchikey14", "exact_mass"]],
    on="inchikey14",
    how="left"
)

mass_test["predicted_mz"] = [
    expected_mz(mass, adduct)
    for mass, adduct in zip(
        mass_test["exact_mass"],
        mass_test["adduct"]
    )
]

mass_test["abs_error_ppm"] = (
    (
        mass_test["precursor_mz"] - mass_test["predicted_mz"]
    )
    / mass_test["predicted_mz"]
    * 1e6
).abs()

valid_mass = (
    mass_test["predicted_mz"].notna()
    & np.isfinite(mass_test["predicted_mz"])
    & (mass_test["predicted_mz"] > 0)
    & np.isfinite(mass_test["abs_error_ppm"])
)

print("Sampled spectra:", len(mass_test))
print("Valid mass calculations:", int(valid_mass.sum()))

for ppm in [5, 10, 20, 50, 100]:
    passed = valid_mass & (mass_test["abs_error_ppm"] <= ppm)
    print(f"Within ±{ppm:>3} ppm: {passed.mean() * 100:.2f}%")

In [ ]:
# Count candidate structures within each precursor-mass tolerance.
# Use chunks to avoid building a huge spectra-by-library matrix.

mass_values = library["exact_mass"].to_numpy()
library_labels = library["inchikey14"].to_numpy()

candidate_counts = {ppm: [] for ppm in [5, 10, 20, 50]}

for row in mass_test.itertuples(index=False):
    observed_mz = row.precursor_mz
    adduct = row.adduct

    # Invert the adduct equation to estimate neutral mass.
    # expected_mz(neutral_mass, adduct) is the forward equation.
    def neutral_mass_from_mz(mz, adduct):
        if adduct in ["[2M+H]+", "[2M-H]-"]:
            offset = PROTON if adduct == "[2M+H]+" else -PROTON
            return (mz - offset) / 2
        if adduct == "[2M+Na]+":
            return (mz - SODIUM) / 2
        if adduct == "[2M+NH4]+":
            return (mz - AMMONIUM) / 2
        if adduct == "[2M+CH2O2-H]-":
            return (mz - FORMIC_ACID_MINUS_H) / 2
        if adduct == "[M+2H]2+":
            return mz * 2 - 2 * PROTON
        if adduct == "[M+3H]3+":
            return mz * 3 - 3 * PROTON

        offsets = {
            "[M+H]+": PROTON,
            "[M-H]-": -PROTON,
            "[M+Na]+": SODIUM,
            "[M+K]+": POTASSIUM,
            "[M+NH4]+": AMMONIUM,
            "[M+Cl]-": CHLORIDE,
            "[M+CH2O2-H]-": FORMIC_ACID_MINUS_H,
            "[M]+": 0.0,
            "[M]-": 0.0,
            "[M-H2O+H]+": PROTON - WATER,
            "[M-2H2O+H]+": PROTON - 2 * WATER,
            "[M-H2O]+": -WATER,
            "[M-H2O-H]-": -WATER - PROTON,
            "[M-2H]-": -2 * PROTON,
            "[M-CH3]-": -15.023475,
        }

        if adduct not in offsets:
            return np.nan
        return mz - offsets[adduct]

    neutral_mass = neutral_mass_from_mz(observed_mz, adduct)

    if not np.isfinite(neutral_mass) or neutral_mass <= 0:
        for ppm in candidate_counts:
            candidate_counts[ppm].append(0)
        continue

    # Approximate neutral-mass tolerance, in ppm.
    errors_ppm = np.abs(mass_values - neutral_mass) / neutral_mass * 1e6

    for ppm in candidate_counts:
        candidate_counts[ppm].append(int(np.sum(errors_ppm <= ppm)))

for ppm, counts in candidate_counts.items():
    counts = np.asarray(counts)
    print(
        f"±{ppm:>2} ppm | "
        f"median candidates: {np.median(counts):.0f} | "
        f"mean: {np.mean(counts):.1f} | "
        f"max: {np.max(counts)} | "
        f"zero candidates: {(counts == 0).sum()}"
    )

In [ ]:
# Measure true-structure retention using the same mass inversion
# as the candidate-count calculation.

retention_results = []

for i, row in enumerate(mass_test.itertuples(index=False)):
    neutral_mass = neutral_mass_from_mz(row.precursor_mz, row.adduct)

    result = {
        "inchikey14": row.inchikey14,
        "adduct": row.adduct,
        "neutral_mass": neutral_mass,
    }

    if not np.isfinite(neutral_mass) or neutral_mass <= 0:
        for ppm in [5, 10, 20, 50]:
            result[f"retained_{ppm}ppm"] = False
            result[f"count_{ppm}ppm"] = 0
        retention_results.append(result)
        continue

    errors_ppm = np.abs(mass_values - neutral_mass) / neutral_mass * 1e6

    true_mass_rows = library["inchikey14"].to_numpy() == row.inchikey14
    true_mass_errors = errors_ppm[true_mass_rows]

    for ppm in [5, 10, 20, 50]:
        result[f"retained_{ppm}ppm"] = bool(
            len(true_mass_errors) > 0 and np.min(true_mass_errors) <= ppm
        )
        result[f"count_{ppm}ppm"] = int(np.sum(errors_ppm <= ppm))

    retention_results.append(result)

retention_df = pd.DataFrame(retention_results)

for ppm in [5, 10, 20, 50]:
    print(
        f"±{ppm:>2} ppm | "
        f"true structure retained: "
        f"{retention_df[f'retained_{ppm}ppm'].mean() * 100:.2f}% | "
        f"median candidates: "
        f"{retention_df[f'count_{ppm}ppm'].median():.0f}"
    )

In [ ]:
print("Queries:", len(queries))
print("Reference structures:", len(refs_all))
print("Query matrix:", Q.shape)
print("Reference matrix:", R.shape)
print("Reference labels:", len(refs_all["inchikey14"]))

print("\nRequired columns:")
print("queries:", {"inchikey14", "precursor_mz", "adduct", "ionization_mode"}.issubset(queries.columns))
print("refs_all:", {"inchikey14", "precursor_mz", "adduct", "ionization_mode"}.issubset(refs_all.columns))

In [ ]:

import numpy as np
import pandas as pd

from sklearn.metrics.pairwise import cosine_similarity

# Confirm that reference labels and similarity-matrix rows align.
assert R.shape[0] == len(refs_all)
assert Q.shape[0] == len(queries)
assert len(refs_all["inchikey14"]) == R.shape[0]

query_labels = queries["inchikey14"].astype(str).to_numpy()
reference_labels = refs_all["inchikey14"].astype(str).to_numpy()

query_mz = queries["precursor_mz"].to_numpy(dtype=float)
reference_mz = refs_all["precursor_mz"].to_numpy(dtype=float)

print("Alignment checks passed.")
print("Queries:", len(query_labels))
print("Reference structures:", len(reference_labels))


In [ ]:

from scipy.sparse import csr_matrix

# R and Q are already L2-normalized sparse fragment feature matrices.
# Compute cosine similarity for manageable query batches.
batch_size = 50
mass_tolerances = [10, 20, 50]
methods = ["mass_only", "fragments_only", "combined"]

results = []

for start in range(0, len(queries), batch_size):
    end = min(start + batch_size, len(queries))

    # Sparse matrix multiplication; each row is a query.
    similarity_batch = (Q[start:end] @ R.T).toarray()

    for local_i, q_idx in enumerate(range(start, end)):
        q_label = query_labels[q_idx]
        q_mz = query_mz[q_idx]

        # Symmetric ppm error based on query precursor m/z.
        ppm_error = np.abs(reference_mz - q_mz) / max(abs(q_mz), 1e-12) * 1e6
        sim = similarity_batch[local_i]

        # Evaluate only candidates within the chosen mass tolerance
        # for combined ranking; mass-only uses the same candidate sets.
        for tol in mass_tolerances:
            eligible = ppm_error <= tol

            for method in methods:
                if method == "fragments_only":
                    scores = sim.copy()
                elif method == "mass_only":
                    scores = -ppm_error
                else:
                    # Rank primarily by fragment similarity, with a small
                    # mass-error preference as a tie-breaker.
                    scores = sim - 0.001 * (ppm_error / tol)

                scores = scores.copy()
                scores[~eligible] = -np.inf

                top_idx = np.argsort(scores)[::-1][:25]
                top_labels = reference_labels[top_idx]
                hits = np.flatnonzero(top_labels == q_label)

                reciprocal_rank = 1.0 / (hits[0] + 1) if len(hits) else 0.0
                hit_at_25 = int(len(hits) > 0)

                results.append({
                    "query_idx": q_idx,
                    "tolerance_ppm": tol,
                    "method": method,
                    "rr_at_25": reciprocal_rank,
                    "hit_at_25": hit_at_25,
                    "candidate_count": int(eligible.sum()),
                })

    if (start // batch_size) % 5 == 0:
        print(f"Processed {end}/{len(queries)} queries")

benchmark = pd.DataFrame(results)

summary = (
    benchmark.groupby(["tolerance_ppm", "method"], as_index=False)
    .agg(
        MRR_at_25=("rr_at_25", "mean"),
        Hit_at_25=("hit_at_25", "mean"),
        Median_candidates=("candidate_count", "median"),
    )
)

display(summary.round(4))


In [ ]:

# Step 1: Check the columns needed for adduct-aware candidate retrieval.

print("Library shape:", library.shape)
print("Library columns:", library.columns.tolist())
print("Query columns:", queries.columns.tolist())

print("\nExample candidate structures:")
display(library.head(5))

print("\nAdduct counts in validation queries:")
display(queries["adduct"].value_counts().head(15))


In [ ]:

# Check how well each adduct equation predicts the known
# structure's observed precursor m/z.

from collections import Counter
import numpy as np
import pandas as pd

mass_check = queries[
    ["inchikey14", "precursor_mz", "adduct"]
].merge(
    library[["inchikey14", "exact_mass"]],
    on="inchikey14",
    how="left"
)

mass_check["predicted_mz"] = [
    expected_mz(mass, adduct)
    if pd.notna(mass) else np.nan
    for mass, adduct in zip(
        mass_check["exact_mass"],
        mass_check["adduct"]
    )
]

mass_check["abs_error_ppm"] = (
    (mass_check["precursor_mz"] - mass_check["predicted_mz"])
    / mass_check["predicted_mz"]
    * 1e6
).abs()

mass_check["valid"] = (
    np.isfinite(mass_check["predicted_mz"])
    & (mass_check["predicted_mz"] > 0)
    & np.isfinite(mass_check["abs_error_ppm"])
)

audit = (
    mass_check.groupby("adduct", dropna=False)
    .agg(
        spectra=("inchikey14", "size"),
        valid_equations=("valid", "sum"),
        median_abs_ppm=("abs_error_ppm", "median"),
        pct_within_10ppm=(
            "abs_error_ppm",
            lambda x: (x <= 10).mean() * 100
        )
    )
    .sort_values("spectra", ascending=False)
)

display(audit.round(2))
print("\nAdducts with no valid equation:")
print(audit.index[audit["valid_equations"] == 0].tolist())


In [ ]:

# Inspect adducts that expected_mz() does not currently support.

unsupported_adducts = [
    "[M+C2H4O2-H]-",
    "[2M+Na-2H]-",
    "[2M+K]+",
    "[M-H2]2-",
]

unsupported_cases = mass_check[
    mass_check["adduct"].isin(unsupported_adducts)
][
    [
        "inchikey14",
        "adduct",
        "precursor_mz",
        "exact_mass",
        "normalized_smiles",
    ]
].copy()

# Add the true molecular formula for chemical interpretation.
unsupported_cases = unsupported_cases.merge(
    library[["inchikey14", "molecular_formula"]],
    on="inchikey14",
    how="left"
)

display(unsupported_cases.to_string(index=False))


In [ ]:

unsupported_adducts = [
    "[M+C2H4O2-H]-",
    "[2M+Na-2H]-",
    "[2M+K]+",
    "[M-H2]2-",
]

unsupported_cases = mass_check[
    mass_check["adduct"].isin(unsupported_adducts)
][
    [
        "inchikey14",
        "adduct",
        "precursor_mz",
        "exact_mass",
    ]
].merge(
    library[
        ["inchikey14", "molecular_formula", "normalized_smiles"]
    ],
    on="inchikey14",
    how="left"
)

display(unsupported_cases.to_string(index=False))


In [ ]:

# Add explicit support for the missing adduct labels.
# Keep the existing expected_mz() function intact for now.

POTASSIUM_CATION = 38.963158
SODIUM_CATION = 22.989218
PROTON_MASS = 1.0072764666
HYDROGEN_ATOM = 1.0078250322
ACETIC_ACID = 60.021129

def expected_mz_extended(neutral_mass, adduct):
    if adduct == "[2M+K]+":
        return 2 * neutral_mass + POTASSIUM_CATION

    if adduct == "[2M+Na-2H]-":
        return 2 * neutral_mass + SODIUM_CATION - 2 * PROTON_MASS

    if adduct == "[M+C2H4O2-H]-":
        return neutral_mass + ACETIC_ACID - PROTON_MASS

    if adduct == "[M-H2]2-":
        return (neutral_mass - 2 * HYDROGEN_ATOM) / 2

    return expected_mz(neutral_mass, adduct)


# Verify the new equations on the seven observed cases.
unsupported_cases["predicted_mz_new"] = [
    expected_mz_extended(mass, adduct)
    for mass, adduct in zip(
        unsupported_cases["exact_mass"],
        unsupported_cases["adduct"]
    )
]

unsupported_cases["abs_error_ppm_new"] = (
    (
        unsupported_cases["precursor_mz"]
        - unsupported_cases["predicted_mz_new"]
    )
    / unsupported_cases["predicted_mz_new"]
    * 1e6
).abs()

display(
    unsupported_cases[
        [
            "adduct",
            "precursor_mz",
            "predicted_mz_new",
            "abs_error_ppm_new",
        ]
    ].round(3)
)


In [ ]:

# Inspect the original training records for the [M-H2]2- case.
target_key = "OFBHPPMPBOJXRT"

train_path = "/kaggle/input/datasets/denisketer/enveda-casmi-2026/train.parquet"

import pyarrow.parquet as pq

pf = pq.ParquetFile(train_path)
matches = []

for batch in pf.iter_batches(
    columns=[
        "inchikey14",
        "normalized_smiles",
        "molecular_formula",
        "adduct",
        "precursor_mz",
        "precursor_error_ppm",
    ],
    batch_size=100_000,
):
    frame = batch.to_pandas()
    found = frame[
        (frame["inchikey14"] == target_key)
        & (frame["adduct"] == "[M-H2]2-")
    ]
    if not found.empty:
        matches.append(found)

if matches:
    display(pd.concat(matches, ignore_index=True).head(20))
else:
    print("No matching original training records found.")


In [ ]:
target_key = "OFBHPPMPBOJXRT"

cols = [
    "inchikey14",
    "adduct",
    "precursor_mz",
    "precursor_error_ppm",
    "adduct_orig",
    "ingest_lib",
    "collision_energy_orig",
    "collision_energy_orig_units",
]

matches = []

for batch in pf.iter_batches(columns=cols, batch_size=100_000):
    frame = batch.to_pandas()
    found = frame[
        (frame["inchikey14"] == target_key)
        & (frame["adduct"] == "[M-H2]2-")
    ]
    if not found.empty:
        matches.append(found)

if matches:
    diagnostic = pd.concat(matches, ignore_index=True)
    print("Matching rows:", len(diagnostic))
    display(diagnostic.drop_duplicates().head(30))
    print("\nUnique values per column:")
    display(diagnostic.nunique(dropna=False).to_frame("unique_values"))
else:
    print("No matching rows found.")

In [ ]:
display(
    diagnostic.groupby(
        ["precursor_error_ppm", "ingest_lib"],
        dropna=False
    )
    .size()
    .reset_index(name="row_count")
    .sort_values("precursor_error_ppm")
)

In [ ]:
observed_mz = 230.5
reported_error_ppm = 126.666706

# If ppm error is relative to the theoretical m/z:
theoretical_mz = observed_mz / (1 - reported_error_ppm / 1_000_000)

print(f"Observed m/z:              {observed_mz:.6f}")
print(f"Theoretical m/z implied:   {theoretical_mz:.6f}")
print(f"Our current predicted m/z: {expected_mz_extended(463.074043, '[M-H2]2-'):.6f}")

In [ ]:
# Summarize large precursor errors by adduct.
# Use the original training parquet data, not the candidate library.

error_summary = (
    train_meta.assign(
        abs_error_ppm=train_meta["precursor_error_ppm"].abs()
    )
    .groupby("adduct")
    .agg(
        spectra=("abs_error_ppm", "size"),
        over_50_ppm=("abs_error_ppm", lambda x: (x > 50).sum()),
        over_100_ppm=("abs_error_ppm", lambda x: (x > 100).sum()),
        median_abs_ppm=("abs_error_ppm", "median"),
    )
    .sort_values("over_100_ppm", ascending=False)
)

display(error_summary.head(15))

In [ ]:
import numpy as np
import pandas as pd

adducts_to_check = ["[M+CH2O2-H]-", "[M-H]-"]

for adduct in adducts_to_check:
    errors = train_meta.loc[
        train_meta["adduct"].eq(adduct),
        "precursor_error_ppm"
    ].dropna().abs()

    print(f"\n{adduct} — n={len(errors):,}")
    print(
        errors.quantile(
            [0, 0.5, 0.9, 0.95, 0.99, 0.999, 1]
        ).to_string()
    )
    print(f"Over 10 ppm:  {(errors > 10).mean():.2%}")
    print(f"Over 50 ppm:  {(errors > 50).mean():.2%}")
    print(f"Over 100 ppm: {(errors > 100).mean():.2%}")

In [ ]:
# Compare formate-adduct mass errors by source library.
formate = train_meta.loc[
    train_meta["adduct"].eq("[M+CH2O2-H]-"),
    ["inchikey14", "adduct", "precursor_mz",
     "precursor_error_ppm"]
].copy()

# Recover the source-library label from the original parquet.
source_rows = []

for batch in pf.iter_batches(
    columns=["inchikey14", "adduct", "precursor_mz",
             "precursor_error_ppm", "ingest_lib"],
    batch_size=100_000
):
    part = batch.to_pandas()
    part = part[part["adduct"].eq("[M+CH2O2-H]-")]
    if not part.empty:
        source_rows.append(part)

formate_sources = pd.concat(source_rows, ignore_index=True)
formate_sources["abs_error_ppm"] = (
    formate_sources["precursor_error_ppm"].abs()
)

summary = (
    formate_sources.groupby("ingest_lib")["abs_error_ppm"]
    .agg(
        spectra="size",
        median="median",
        p90=lambda x: x.quantile(0.90),
        over_100ppm=lambda x: (x > 100).mean()
    )
    .sort_values("over_100ppm", ascending=False)
)

display(summary)

In [ ]:
# Sample formate-adduct records from the two contrasting sources.
sample_rows = (
    formate_sources[
        formate_sources["ingest_lib"].isin(["pluskal_ms2", "mona"])
    ]
    .groupby("ingest_lib", group_keys=False)
    .sample(n=5, random_state=42)
    .copy()
)

sample_rows = sample_rows.merge(
    library[["inchikey14", "exact_mass"]].drop_duplicates("inchikey14"),
    on="inchikey14",
    how="left"
)

sample_rows["calculated_mz"] = sample_rows.apply(
    lambda row: expected_mz_extended(
        row["exact_mass"], row["adduct"]
    ),
    axis=1
)

sample_rows["recomputed_error_ppm"] = (
    (sample_rows["precursor_mz"] - sample_rows["calculated_mz"])
    / sample_rows["calculated_mz"] * 1_000_000
)

display(
    sample_rows[
        [
            "ingest_lib", "inchikey14", "precursor_mz",
            "exact_mass", "calculated_mz",
            "precursor_error_ppm", "recomputed_error_ppm"
        ]
    ].sort_values("ingest_lib")
)

In [ ]:
# Compare stored versus independently recomputed ppm error
# for a random sample of formate-adduct records.

check = (
    formate_sources
    .groupby("ingest_lib", group_keys=False)
    .apply(
        lambda g: g.sample(
            n=min(500, len(g)),
            random_state=42
        ),
        include_groups=True
    )
    .reset_index(drop=True)
)

check = check.merge(
    library[["inchikey14", "exact_mass"]].drop_duplicates("inchikey14"),
    on="inchikey14",
    how="left"
)

check["calculated_mz"] = check.apply(
    lambda r: expected_mz_extended(r["exact_mass"], r["adduct"]),
    axis=1
)

check["recomputed_error_ppm"] = (
    (check["precursor_mz"] - check["calculated_mz"])
    / check["calculated_mz"] * 1_000_000
)

check["stored_abs_error"] = check["precursor_error_ppm"].abs()
check["recomputed_abs_error"] = check["recomputed_error_ppm"].abs()

display(
    check.groupby("ingest_lib").agg(
        sampled_rows=("inchikey14", "size"),
        stored_median_abs_ppm=("stored_abs_error", "median"),
        recomputed_median_abs_ppm=("recomputed_abs_error", "median"),
        recomputed_pct_within_10ppm=(
            "recomputed_abs_error",
            lambda x: (x <= 10).mean() * 100
        )
    ).sort_values("sampled_rows", ascending=False)
)

In [ ]:
# Sample up to 300 records per common adduct and source.
# Recompute ppm errors independently of precursor_error_ppm.

common_adducts = ["[M+H]+", "[M-H]-", "[M+Na]+", "[M+CH2O2-H]-"]

check_common = []

for batch in pf.iter_batches(
    columns=["inchikey14", "adduct", "precursor_mz", "ingest_lib"],
    batch_size=100_000
):
    part = batch.to_pandas()
    part = part[part["adduct"].isin(common_adducts)]
    if not part.empty:
        check_common.append(part)

check_common = pd.concat(check_common, ignore_index=True)

check_common = (
    check_common
    .groupby(["adduct", "ingest_lib"], group_keys=False)
    .apply(
        lambda g: g.sample(n=min(300, len(g)), random_state=42)
    )
    .reset_index(drop=True)
)

check_common = check_common.merge(
    library[["inchikey14", "exact_mass"]].drop_duplicates("inchikey14"),
    on="inchikey14",
    how="left"
)

check_common["calculated_mz"] = check_common.apply(
    lambda r: expected_mz_extended(r["exact_mass"], r["adduct"]),
    axis=1
)

check_common["abs_recomputed_ppm"] = (
    (check_common["precursor_mz"] - check_common["calculated_mz"]).abs()
    / check_common["calculated_mz"] * 1_000_000
)

display(
    check_common.groupby(["adduct", "ingest_lib"]).agg(
        n=("inchikey14", "size"),
        median_abs_ppm=("abs_recomputed_ppm", "median"),
        pct_within_10ppm=(
            "abs_recomputed_ppm",
            lambda x: (x <= 10).mean() * 100
        )
    ).sort_values(["adduct", "pct_within_10ppm"])
)

In [1]:

from pathlib import Path
from datetime import datetime

checkpoint = r"""
ENVEDA CASMI 2026 — PROJECT CHECKPOINT
Purpose: Resume the Kaggle molecule identification project.

DATA
- Dataset: /kaggle/input/datasets/denisketer/enveda-casmi-2026/
- Files: train.parquet, test.parquet, sample_submission.csv
- Train: 2,539,608 spectra rows
- Test: 1,213 spectra across 400 molecule IDs
- Candidate library: 275,810 unique InChIKey14 structures

CURRENT APPROACH
- Build candidate library from training structures using RDKit.
- Use molecular exact mass and adduct information for candidate filtering.
- Use MS/MS fragment similarity to rank candidates.
- Produce up to 25 semicolon-separated SMILES per molecule_id.
- Evaluation target: MRR@25 after tautomer canonicalization.

IMPORTANT NOTEBOOK OBJECTS
- library: 275,810 rows; normalized_smiles, inchikey14,
  molecular_formula, exact_mass, rdkit_formula
- train_meta: precursor/adduct metadata from training data
- test_meta: test molecule and precursor metadata
- refs_all: one reference spectrum per structure
- queries: 2,000 sampled validation spectra
- Q and R: sparse query/reference fragment-feature matrices
- expected_mz and expected_mz_extended: adduct mass functions
- submission_baseline.csv: format smoke test only, not a competitive model

VALIDATION WORK
- Mass-only retrieval needs adduct-aware mass calculations.
- Preliminary fragment retrieval is a diagnostic, not competition performance.
- Do not interpret earlier biased or preliminary benchmark scores as final results.
- The stored precursor_error_ppm can be unreliable for some source libraries.
- For [M+CH2O2-H]-, pluskal_ms2 and gnps have very large stored errors,
  but sampled records have small independently recomputed mass errors.
- Do not use stored precursor_error_ppm as a hard candidate filter.
- [M-H2]2- has an unresolved observed/theoretical mass discrepancy.

NEXT STEP
1. Inspect independently recomputed mass errors for common adducts:
   [M+H]+, [M-H]-, [M+Na]+, [M+CH2O2-H]-.
2. Then decide evidence-based mass tolerances and fallback handling.
3. Validate candidate retrieval before building the final submission.

SAFETY
- Preserve the current notebook and do not overwrite the original train data.
- Recheck variable availability after restarting the session.
- Recreate objects from the original parquet files if necessary.
"""

path = Path("/kaggle/working/progress_checkpoint.txt")
path.write_text(
    f"Checkpoint saved: {datetime.now().isoformat(timespec='minutes')}\n"
    + checkpoint,
    encoding="utf-8"
)

print("Checkpoint saved:", path)
print("File size:", path.stat().st_size, "bytes")


Checkpoint saved: /kaggle/working/progress_checkpoint.txt
File size: 2336 bytes


In [2]:
from pathlib import Path

checkpoint = Path("/kaggle/working/progress_checkpoint.txt")

print("Checkpoint exists:", checkpoint.exists())

if checkpoint.exists():
    print("File size:", checkpoint.stat().st_size, "bytes")
    print("\nLast 1000 characters:\n")
    print(checkpoint.read_text()[-1000:])

Checkpoint exists: True
File size: 2336 bytes

Last 1000 characters:

tions.
- Preliminary fragment retrieval is a diagnostic, not competition performance.
- Do not interpret earlier biased or preliminary benchmark scores as final results.
- The stored precursor_error_ppm can be unreliable for some source libraries.
- For [M+CH2O2-H]-, pluskal_ms2 and gnps have very large stored errors,
  but sampled records have small independently recomputed mass errors.
- Do not use stored precursor_error_ppm as a hard candidate filter.
- [M-H2]2- has an unresolved observed/theoretical mass discrepancy.

NEXT STEP
1. Inspect independently recomputed mass errors for common adducts:
   [M+H]+, [M-H]-, [M+Na]+, [M+CH2O2-H]-.
2. Then decide evidence-based mass tolerances and fallback handling.
3. Validate candidate retrieval before building the final submission.

SAFETY
- Preserve the current notebook and do not overwrite the original train data.
- Recheck variable availability after restarting the sess

In [3]:
check_names = [
    "train", "test", "library",
    "val_spectra", "validation_spectra",
    "reference_spectra", "refs_all",
    "queries", "expected_mz", "expected_mz_extended"
]

for name in check_names:
    exists = name in globals()
    print(f"{name}: {'Available' if exists else 'Not available'}")

train: Not available
test: Not available
library: Not available
val_spectra: Not available
validation_spectra: Not available
reference_spectra: Not available
refs_all: Not available
queries: Not available
expected_mz: Not available
expected_mz_extended: Not available


In [4]:
from pathlib import Path

data_dir = Path("/kaggle/input/datasets/denisketer/enveda-casmi-2026/")

print("Dataset folder exists:", data_dir.exists())

if data_dir.exists():
    for file in data_dir.iterdir():
        print(file.name)

Dataset folder exists: True
sample_submission.csv
train.parquet
test.parquet


In [5]:
import pandas as pd
from pathlib import Path

data_dir = Path("/kaggle/input/datasets/denisketer/enveda-casmi-2026/")

test = pd.read_parquet(data_dir / "test.parquet")
sample_submission = pd.read_csv(data_dir / "sample_submission.csv")

print("Test shape:", test.shape)
print("Test columns:", test.columns.tolist())

print("\nSample submission shape:", sample_submission.shape)
print("Sample submission columns:", sample_submission.columns.tolist())

print("\nFirst 3 submission rows:")
display(sample_submission.head(3))

Test shape: (1213, 12)
Test columns: ['molecule_id', 'spectrum_id', 'ms2_mzs', 'ms2_normalized_intensities', 'base_peak_intensity', 'adduct', 'ionization_mode', 'instrument_type', 'precursor_mz', 'collision_energy_orig', 'collision_energy_ev', 'collision_energy_orig_units']

Sample submission shape: (400, 2)
Sample submission columns: ['molecule_id', 'smiles']

First 3 submission rows:


,molecule_id,smiles
0,m_005e53,CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CC...
1,m_006153,CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CC...
2,m_00b5aa,CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CC...


In [6]:
try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors

    mol = Chem.MolFromSmiles("CCO")

    print("RDKit: available")
    print("Test molecule: ethanol")
    print("Valid SMILES:", Chem.MolToSmiles(mol))
    print("Exact mass:", round(Descriptors.ExactMolWt(mol), 4))

except ImportError:
    print("RDKit: not installed in this session")

RDKit: not installed in this session


In [7]:
%pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.8/31.8 MB 50.7 MB/s eta 0:00:00:00:0100:01
Note: you may need to restart the kernel to use updated packages.


In [8]:
try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors

    mol = Chem.MolFromSmiles("CCO")

    print("RDKit: available")
    print("Test molecule: ethanol")
    print("Valid SMILES:", Chem.MolToSmiles(mol))
    print("Exact mass:", round(Descriptors.ExactMolWt(mol), 4))

except ImportError:
    print("RDKit: not installed in this session")

RDKit: available
Test molecule: ethanol
Valid SMILES: CCO
Exact mass: 46.0419


In [9]:
import pyarrow.parquet as pq
from pathlib import Path

train_path = Path(
    "/kaggle/input/datasets/denisketer/enveda-casmi-2026/train.parquet"
)

parquet_file = pq.ParquetFile(train_path)

print("Training rows:", parquet_file.metadata.num_rows)
print("Row groups:", parquet_file.num_row_groups)
print("\nAvailable columns:")
print(parquet_file.schema.names)

Training rows: 2539608
Row groups: 21

Available columns:
['ingest_lib', 'normalized_smiles', 'inchikey', 'inchikey14', 'molecular_formula', 'ionization_mode', 'instrument_type', 'adduct', 'adduct_orig', 'precursor_mz', 'precursor_error_ppm', 'element', 'element', 'num_peaks', 'base_peak_intensity', 'element', 'collision_energy_orig', 'collision_energy_orig_units']


In [10]:
print(parquet_file.schema)

required group field_id=-1 schema {
  optional binary field_id=-1 ingest_lib (String);
  optional binary field_id=-1 normalized_smiles (String);
  optional binary field_id=-1 inchikey (String);
  optional binary field_id=-1 inchikey14 (String);
  optional binary field_id=-1 molecular_formula (String);
  optional binary field_id=-1 ionization_mode (String);
  optional binary field_id=-1 instrument_type (String);
  optional binary field_id=-1 adduct (String);
  optional binary field_id=-1 adduct_orig (String);
  optional double field_id=-1 precursor_mz;
  optional double field_id=-1 precursor_error_ppm;
  optional group field_id=-1 ms2_mzs (List) {
    repeated group field_id=-1 list {
      optional double field_id=-1 element;
    }
  }
  optional group field_id=-1 ms2_normalized_intensities (List) {
    repeated group field_id=-1 list {
      optional double field_id=-1 element;
    }
  }
  optional int64 field_id=-1 num_peaks;
  optional double field_id=-1 base_peak_intensity;
  optio

In [11]:
import pandas as pd

identity_columns = [
    "normalized_smiles",
    "inchikey14",
    "molecular_formula",
    "ingest_lib",
]

identity_data = pd.read_parquet(
    train_path,
    columns=identity_columns
)

print("Rows loaded:", len(identity_data))
print("Unique structures:", identity_data["inchikey14"].nunique())
print("Missing InChIKey14:", identity_data["inchikey14"].isna().sum())
print("\nFirst 5 rows:")
display(identity_data.head())

Rows loaded: 2539608
Unique structures: 275810
Missing InChIKey14: 0

First 5 rows:


,normalized_smiles,inchikey14,molecular_formula,ingest_lib
0,O=C1NC(=O)c2cc(Nc3ccccc3)c(Nc3ccccc3)cc21,AAALVYBICLMAMA,C20H15N3O2,drug_plus
1,C#Cc1cccc(Nc2ncnc3cc(OCCOC)c(OCCOC)cc23)c1,AAKJLRGGTJKAMG,C22H23N3O4,drug_plus
2,CC1(C)CCC(C)(C)c2cc(C(O)C(O)=Nc3ccc(C(=O)O)cc3...,AANFHDFOMFRLLR,C23H26FNO4,drug_plus
3,CN1C(=O)CN=C(c2ccccc2)c2cc(Cl)ccc21,AAOVKJBEBIDNHE,C16H13ClN2O,drug_plus
4,NC(=O)C1=CN(C2OC(COP(=O)(O)OP(=O)(O)OCC3OC(n4c...,ACFIXJIJDZMPPO,C21H30N7O17P3,drug_plus


In [12]:
candidate_table = (
    identity_data
    .dropna(subset=["normalized_smiles", "inchikey14"])
    .drop_duplicates(subset=["inchikey14"])
    .reset_index(drop=True)
)

print("Unique candidates:", len(candidate_table))
print("Duplicate identities remaining:",
      candidate_table["inchikey14"].duplicated().sum())

display(candidate_table.head(5))

Unique candidates: 275810
Duplicate identities remaining: 0


,normalized_smiles,inchikey14,molecular_formula,ingest_lib
0,O=C1NC(=O)c2cc(Nc3ccccc3)c(Nc3ccccc3)cc21,AAALVYBICLMAMA,C20H15N3O2,drug_plus
1,C#Cc1cccc(Nc2ncnc3cc(OCCOC)c(OCCOC)cc23)c1,AAKJLRGGTJKAMG,C22H23N3O4,drug_plus
2,CC1(C)CCC(C)(C)c2cc(C(O)C(O)=Nc3ccc(C(=O)O)cc3...,AANFHDFOMFRLLR,C23H26FNO4,drug_plus
3,CN1C(=O)CN=C(c2ccccc2)c2cc(Cl)ccc21,AAOVKJBEBIDNHE,C16H13ClN2O,drug_plus
4,NC(=O)C1=CN(C2OC(COP(=O)(O)OP(=O)(O)OCC3OC(n4c...,ACFIXJIJDZMPPO,C21H30N7O17P3,drug_plus


In [13]:
from rdkit import Chem
from rdkit.Chem import Descriptors
import numpy as np

def calculate_exact_mass(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.nan
    return Descriptors.ExactMolWt(mol)

candidate_table["exact_mass"] = candidate_table[
    "normalized_smiles"
].map(calculate_exact_mass)

print("Candidates:", len(candidate_table))
print("Missing exact masses:", candidate_table["exact_mass"].isna().sum())
print("\nExact mass summary:")
display(candidate_table["exact_mass"].describe())

Candidates: 275810
Missing exact masses: 0

Exact mass summary:


count    275810.000000
mean        358.151396
std         133.001371
min          31.042199
25%         304.178693
50%         333.162331
75%         366.119127
max        3399.572640
Name: exact_mass, dtype: float64

In [14]:
print("Test spectra:", len(test))
print("\nAdduct counts:")
display(test["adduct"].value_counts().to_frame("count"))

print("\nMissing precursor masses:", test["precursor_mz"].isna().sum())
print("Missing adduct labels:", test["adduct"].isna().sum())

Test spectra: 1213

Adduct counts:


,count
adduct,
[M+H]+,959
[M-H]-,193
[M+CH2O2-H]-,31
[M+Na]+,22
[M+NH4]+,4
[M+K]+,2
[M+Cl]-,2



Missing precursor masses: 0
Missing adduct labels: 0


In [15]:
PROTON = 1.0072764666
SODIUM = 22.989218
POTASSIUM = 38.963158
AMMONIUM = 18.033823
CHLORIDE = 34.969402
FORMATE = 44.998201

def expected_test_mz(neutral_mass, adduct):
    offsets = {
        "[M+H]+": PROTON,
        "[M-H]-": -PROTON,
        "[M+Na]+": SODIUM,
        "[M+NH4]+": AMMONIUM,
        "[M+K]+": POTASSIUM,
        "[M+Cl]-": CHLORIDE,
        "[M+CH2O2-H]-": FORMATE,
    }

    if adduct not in offsets:
        return np.nan

    return neutral_mass + offsets[adduct]

# Check a few example candidates against the test adducts
for adduct in test["adduct"].value_counts().index:
    example_mz = expected_test_mz(
        candidate_table.loc[0, "exact_mass"],
        adduct
    )
    print(f"{adduct:16s} expected m/z: {example_mz:.5f}")

[M+H]+           expected m/z: 330.12370
[M-H]-           expected m/z: 328.10915
[M+CH2O2-H]-     expected m/z: 374.11463
[M+Na]+          expected m/z: 352.10564
[M+NH4]+         expected m/z: 347.15025
[M+K]+           expected m/z: 368.07958
[M+Cl]-          expected m/z: 364.08583


In [16]:
mass_values = candidate_table["exact_mass"].to_numpy()

adduct_offsets = {
    "[M+H]+": PROTON,
    "[M-H]-": -PROTON,
    "[M+Na]+": SODIUM,
    "[M+NH4]+": AMMONIUM,
    "[M+K]+": POTASSIUM,
    "[M+Cl]-": CHLORIDE,
    "[M+CH2O2-H]-": FORMATE,
}

tolerances = [5, 10, 20, 50]
results = []

for row in test.itertuples(index=False):
    offset = adduct_offsets[row.adduct]
    predicted_mz = mass_values + offset
    ppm_error = np.abs(predicted_mz - row.precursor_mz) / row.precursor_mz * 1e6

    result = {
        "molecule_id": row.molecule_id,
        "adduct": row.adduct,
    }

    for tol in tolerances:
        result[f"candidates_{tol}ppm"] = int((ppm_error <= tol).sum())

    results.append(result)

mass_coverage = pd.DataFrame(results)

for tol in tolerances:
    counts = mass_coverage[f"candidates_{tol}ppm"]
    print(
        f"±{tol:2d} ppm | "
        f"median candidates: {counts.median():.0f} | "
        f"mean: {counts.mean():.1f} | "
        f"zero candidates: {(counts == 0).sum()} | "
        f"over 25 candidates: {(counts > 25).sum()}"
    )

± 5 ppm | median candidates: 59 | mean: 94.1 | zero candidates: 0 | over 25 candidates: 914
±10 ppm | median candidates: 82 | mean: 118.6 | zero candidates: 0 | over 25 candidates: 1028
±20 ppm | median candidates: 126 | mean: 164.1 | zero candidates: 0 | over 25 candidates: 1148
±50 ppm | median candidates: 322 | mean: 390.9 | zero candidates: 0 | over 25 candidates: 1193


In [17]:
mass_values = candidate_table["exact_mass"].to_numpy()

adduct_offsets = {
    "[M+H]+": PROTON,
    "[M-H]-": -PROTON,
    "[M+Na]+": SODIUM,
    "[M+NH4]+": AMMONIUM,
    "[M+K]+": POTASSIUM,
    "[M+Cl]-": CHLORIDE,
    "[M+CH2O2-H]-": FORMATE,
}

tolerances = [5, 10, 20, 50]
results = []

for row in test.itertuples(index=False):
    offset = adduct_offsets[row.adduct]
    predicted_mz = mass_values + offset
    ppm_error = np.abs(predicted_mz - row.precursor_mz) / row.precursor_mz * 1e6

    result = {
        "molecule_id": row.molecule_id,
        "adduct": row.adduct,
    }

    for tol in tolerances:
        result[f"candidates_{tol}ppm"] = int((ppm_error <= tol).sum())

    results.append(result)

mass_coverage = pd.DataFrame(results)

for tol in tolerances:
    counts = mass_coverage[f"candidates_{tol}ppm"]
    print(
        f"±{tol:2d} ppm | "
        f"median candidates: {counts.median():.0f} | "
        f"mean: {counts.mean():.1f} | "
        f"zero candidates: {(counts == 0).sum()} | "
        f"over 25 candidates: {(counts > 25).sum()}"
    )

± 5 ppm | median candidates: 59 | mean: 94.1 | zero candidates: 0 | over 25 candidates: 914
±10 ppm | median candidates: 82 | mean: 118.6 | zero candidates: 0 | over 25 candidates: 1028
±20 ppm | median candidates: 126 | mean: 164.1 | zero candidates: 0 | over 25 candidates: 1148
±50 ppm | median candidates: 322 | mean: 390.9 | zero candidates: 0 | over 25 candidates: 1193


In [18]:
print("Missing fragment m/z lists:",
      test["ms2_mzs"].isna().sum())

print("Missing intensity lists:",
      test["ms2_normalized_intensities"].isna().sum())

for row in test.head(5).itertuples(index=False):
    mzs = row.ms2_mzs
    intensities = row.ms2_normalized_intensities

    print(
        row.molecule_id,
        "| m/z peaks:", len(mzs) if mzs is not None else 0,
        "| intensity values:", len(intensities) if intensities is not None else 0
    )

Missing fragment m/z lists: 0
Missing intensity lists: 0
m_005e53 | m/z peaks: 183 | intensity values: 183
m_005e53 | m/z peaks: 119 | intensity values: 119
m_005e53 | m/z peaks: 84 | intensity values: 84
m_005e53 | m/z peaks: 33 | intensity values: 33
m_006153 | m/z peaks: 219 | intensity values: 219


In [19]:
validation_columns = [
    "normalized_smiles",
    "inchikey14",
    "ionization_mode",
    "adduct",
    "precursor_mz",
    "ms2_mzs",
    "ms2_normalized_intensities",
]

train_spectra = pd.read_parquet(
    train_path,
    columns=validation_columns
)

print("Training spectra loaded:", len(train_spectra))
print("Unique molecular identities:", train_spectra["inchikey14"].nunique())
print("Missing fragment lists:", train_spectra["ms2_mzs"].isna().sum())

Training spectra loaded: 2539608
Unique molecular identities: 275810
Missing fragment lists: 0


In [20]:
import numpy as np

rng = np.random.default_rng(42)

# Count spectra available for each molecular identity
identity_counts = train_spectra["inchikey14"].value_counts()

# Only identities with at least two spectra can support this check
eligible = train_spectra[
    train_spectra["inchikey14"].map(identity_counts) >= 2
]

print("Eligible spectra:", len(eligible))
print("Eligible identities:", eligible["inchikey14"].nunique())

# Sample a manageable validation set
validation_sample = eligible.sample(
    n=min(500, len(eligible)),
    random_state=42
).copy()

print("Validation sample:", len(validation_sample))
print("Validation identities:", validation_sample["inchikey14"].nunique())

Eligible spectra: 2522575
Eligible identities: 258777
Validation sample: 500
Validation identities: 498


In [21]:
# Use the sampled row indices to keep validation spectra out of references
validation_indices = validation_sample.index

reference_spectra = train_spectra.drop(
    index=validation_indices
)

print("Validation spectra:", len(validation_sample))
print("Reference spectra:", len(reference_spectra))

overlap = set(validation_sample.index).intersection(reference_spectra.index)
print("Row-index overlap:", len(overlap))

Validation spectra: 500
Reference spectra: 2539108
Row-index overlap: 0


In [22]:
validation_ids = set(validation_sample["inchikey14"])
reference_ids = set(reference_spectra["inchikey14"])

represented = validation_ids & reference_ids
missing_ids = validation_ids - reference_ids

print("Validation identities:", len(validation_ids))
print("Represented in references:", len(represented))
print("Missing from references:", len(missing_ids))

Validation identities: 498
Represented in references: 498
Missing from references: 0


In [23]:
refs_all = (
    reference_spectra
    .drop_duplicates(subset="inchikey14", keep="first")
    .reset_index(drop=True)
)

print("Reference spectra:", len(reference_spectra))
print("Unique reference identities:", len(refs_all))
print("Duplicate identities remaining:",
      refs_all["inchikey14"].duplicated().sum())

display(
    refs_all[
        ["inchikey14", "normalized_smiles", "precursor_mz", "adduct"]
    ].head()
)

Reference spectra: 2539108
Unique reference identities: 275810
Duplicate identities remaining: 0


,inchikey14,normalized_smiles,precursor_mz,adduct
0,AAALVYBICLMAMA,O=C1NC(=O)c2cc(Nc3ccccc3)c(Nc3ccccc3)cc21,330.1237,[M+H]+
1,AAKJLRGGTJKAMG,C#Cc1cccc(Nc2ncnc3cc(OCCOC)c(OCCOC)cc23)c1,432.1320,[M+K]+
2,AANFHDFOMFRLLR,CC1(C)CCC(C)(C)c2cc(C(O)C(O)=Nc3ccc(C(=O)O)cc3...,422.1738,[M+Na]+
3,AAOVKJBEBIDNHE,CN1C(=O)CN=C(c2ccccc2)c2cc(Cl)ccc21,285.0789,[M+H]+
4,ACFIXJIJDZMPPO,NC(=O)C1=CN(C2OC(COP(=O)(O)OP(=O)(O)OCC3OC(n4c...,746.0984,[M+H]+


In [24]:
from collections import defaultdict
import numpy as np

def spectrum_to_dict(mzs, intensities, bin_width=0.02):
    features = defaultdict(float)

    for mz, intensity in zip(mzs, intensities):
        if 50 <= mz <= 1000 and intensity > 0:
            bin_id = int(round(mz / bin_width))
            features[bin_id] += np.sqrt(float(intensity))

    return dict(features)

example = validation_sample.iloc[0]

example_features = spectrum_to_dict(
    example["ms2_mzs"],
    example["ms2_normalized_intensities"]
)

print("Molecule identity:", example["inchikey14"])
print("Original peaks:", len(example["ms2_mzs"]))
print("Non-zero fragment features:", len(example_features))
print("First 10 features:", list(example_features.items())[:10])

Molecule identity: OVMFOVNOXASTPA
Original peaks: 48
Non-zero fragment features: 48
First 10 features: [(16254, np.float64(0.161245154965971)), (16854, np.float64(0.11832159566199232)), (17655, np.float64(0.1673320053068151)), (17754, np.float64(0.17320508075688773)), (18656, np.float64(0.14491376746189438)), (18954, np.float64(0.22583179581272428)), (19554, np.float64(0.10954451150103323)), (19855, np.float64(0.3492849839314596)), (20455, np.float64(0.406201920231798)), (20755, np.float64(0.3301514803843836))]


In [25]:
print("Validation spectrum metadata:")
print("InChIKey14:", example["inchikey14"])
print("Adduct:", example["adduct"])
print("Ionization mode:", example["ionization_mode"])
print("Precursor m/z:", example["precursor_mz"])

print("\nReference library metadata available:")
print("Adduct column:", "adduct" in refs_all.columns)
print("Ionization mode column:", "ionization_mode" in refs_all.columns)
print("Precursor m/z column:", "precursor_mz" in refs_all.columns)

Validation spectrum metadata:
InChIKey14: OVMFOVNOXASTPA
Adduct: [M+H]+
Ionization mode: positive
Precursor m/z: 565.156

Reference library metadata available:
Adduct column: True
Ionization mode column: True
Precursor m/z column: True


In [26]:
query_mode = example["ionization_mode"]
query_mz = float(example["precursor_mz"])

same_mode = refs_all[
    refs_all["ionization_mode"] == query_mode
].copy()

same_mode["precursor_error_ppm"] = (
    (same_mode["precursor_mz"] - query_mz).abs()
    / query_mz * 1e6
)

print("Query mode:", query_mode)
print("References in same mode:", len(same_mode))

for tol in [5, 10, 20, 50]:
    count = (same_mode["precursor_error_ppm"] <= tol).sum()
    print(f"Within ±{tol} ppm using observed reference m/z: {count}")

Query mode: positive
References in same mode: 236161
Within ±5 ppm using observed reference m/z: 11
Within ±10 ppm using observed reference m/z: 12
Within ±20 ppm using observed reference m/z: 19
Within ±50 ppm using observed reference m/z: 32


In [27]:
query_adduct = example["adduct"]
query_mz = float(example["precursor_mz"])

if query_adduct == "[M+H]+":
    query_neutral_mass = query_mz - PROTON
elif query_adduct == "[M-H]-":
    query_neutral_mass = query_mz + PROTON
elif query_adduct == "[M+Na]+":
    query_neutral_mass = query_mz - SODIUM
elif query_adduct == "[M+NH4]+":
    query_neutral_mass = query_mz - AMMONIUM
elif query_adduct == "[M+K]+":
    query_neutral_mass = query_mz - POTASSIUM
elif query_adduct == "[M+Cl]-":
    query_neutral_mass = query_mz - CHLORIDE
elif query_adduct == "[M+CH2O2-H]-":
    query_neutral_mass = query_mz - FORMATE
else:
    query_neutral_mass = np.nan

print("Query adduct:", query_adduct)
print("Observed precursor m/z:", query_mz)
print("Implied neutral mass:", round(query_neutral_mass, 6))

Query adduct: [M+H]+
Observed precursor m/z: 565.156
Implied neutral mass: 564.148724


In [28]:
candidate_masses = candidate_table["exact_mass"].to_numpy()

ppm_errors = (
    np.abs(candidate_masses - query_neutral_mass)
    / query_neutral_mass * 1e6
)

print("Query neutral mass:", round(query_neutral_mass, 6))

for tol in [5, 10, 20, 50]:
    matched = ppm_errors <= tol
    print(
        f"±{tol:2d} ppm: {int(matched.sum())} candidates"
    )

best_idx = np.argsort(ppm_errors)[:5]

print("\nFive closest candidates:")
display(
    candidate_table.iloc[best_idx][
        ["inchikey14", "normalized_smiles", "exact_mass"]
    ].assign(
        ppm_error=ppm_errors[best_idx]
    )
)

Query neutral mass: 564.148724
± 5 ppm: 16 candidates
±10 ppm: 17 candidates
±20 ppm: 17 candidates
±50 ppm: 19 candidates

Five closest candidates:


,inchikey14,normalized_smiles,exact_mass,ppm_error
191783,DQBVFTJNUYZVQL,CC1OC(Oc2cc(O)c3c(=O)c(OC4OCC(O)C(O)C4O)c(-c4c...,564.147906,1.449897
229813,LOCBOCXGFMHGIJ,O=c1cc(-c2ccc(O)cc2)oc2cc(O)c(C3OCC(O)C(O)C3OC...,564.147906,1.449897
217257,TUIJPUWSXVFWSH,O=c1cc(-c2ccc(O)cc2)oc2c(C3OC(CO)C(O)C3O)c(O)c...,564.147906,1.449897
217907,UDSFTSASYKOZQX,COc1c(OC2OCC(O)C(OC3OCC(O)C(O)C3O)C2O)c(O)c2oc...,564.147906,1.449897
191878,DRLZZQRQMWQRLZ,O=c1cc(-c2ccc(O)cc2)oc2c(C3OC(CO)C(O)C(O)C3O)c...,564.147906,1.449897


In [29]:
query_id = example["inchikey14"]

query_features = spectrum_to_dict(
    example["ms2_mzs"],
    example["ms2_normalized_intensities"]
)

# Find a reference spectrum with the same identity,
# excluding the validation row itself.
same_identity_refs = reference_spectra[
    (reference_spectra["inchikey14"] == query_id)
    & (reference_spectra.index != example.name)
]

print("Query identity:", query_id)
print("Separate reference spectra for this identity:",
      len(same_identity_refs))

if len(same_identity_refs) > 0:
    ref_example = same_identity_refs.iloc[0]

    ref_features = spectrum_to_dict(
        ref_example["ms2_mzs"],
        ref_example["ms2_normalized_intensities"]
    )

    common_bins = set(query_features) & set(ref_features)

    dot_product = sum(
        query_features[k] * ref_features[k]
        for k in common_bins
    )

    query_norm = np.sqrt(sum(v * v for v in query_features.values()))
    ref_norm = np.sqrt(sum(v * v for v in ref_features.values()))

    cosine = dot_product / (query_norm * ref_norm) if query_norm and ref_norm else 0.0

    print("Query fragment features:", len(query_features))
    print("Reference fragment features:", len(ref_features))
    print("Shared fragment bins:", len(common_bins))
    print("Cosine similarity:", round(cosine, 4))
else:
    print("No separate reference spectrum found.")

Query identity: OVMFOVNOXASTPA
Separate reference spectra for this identity: 349
Query fragment features: 48
Reference fragment features: 104
Shared fragment bins: 21
Cosine similarity: 0.5791


In [30]:
# Select 100 reference spectra from different identities
different_refs = refs_all[
    refs_all["inchikey14"] != query_id
].sample(n=100, random_state=42)

query_norm = np.sqrt(sum(v * v for v in query_features.values()))

scores = []

for row in different_refs.itertuples(index=False):
    ref_features = spectrum_to_dict(
        row.ms2_mzs,
        row.ms2_normalized_intensities
    )

    common = set(query_features) & set(ref_features)

    dot = sum(
        query_features[k] * ref_features[k]
        for k in common
    )

    ref_norm = np.sqrt(sum(v * v for v in ref_features.values()))
    cosine = dot / (query_norm * ref_norm) if query_norm and ref_norm else 0.0

    scores.append(cosine)

print("Different-identity comparisons:", len(scores))
print("Median cosine:", round(float(np.median(scores)), 4))
print("95th percentile:", round(float(np.percentile(scores, 95)), 4))
print("Maximum cosine:", round(float(np.max(scores)), 4))
print("Same-identity reference cosine:", 0.5791)

Different-identity comparisons: 100
Median cosine: 0.0
95th percentile: 0.0083
Maximum cosine: 0.0218
Same-identity reference cosine: 0.5791


In [31]:
rng = np.random.default_rng(42)

validation_examples = validation_sample.sample(
    n=20, random_state=42
)

results = []

for _, query in validation_examples.iterrows():
    q_features = spectrum_to_dict(
        query["ms2_mzs"],
        query["ms2_normalized_intensities"]
    )
    q_norm = np.sqrt(sum(v * v for v in q_features.values()))

    # One separate spectrum with the same identity
    same_refs = reference_spectra[
        reference_spectra["inchikey14"] == query["inchikey14"]
    ]

    if len(same_refs) == 0 or q_norm == 0:
        continue

    same_ref = same_refs.iloc[0]
    same_features = spectrum_to_dict(
        same_ref["ms2_mzs"],
        same_ref["ms2_normalized_intensities"]
    )
    same_norm = np.sqrt(sum(v * v for v in same_features.values()))
    common = set(q_features) & set(same_features)

    same_score = (
        sum(q_features[k] * same_features[k] for k in common)
        / (q_norm * same_norm)
        if same_norm else 0.0
    )

    # 100 randomly selected different identities
    other_refs = refs_all[
        refs_all["inchikey14"] != query["inchikey14"]
    ].sample(n=100, random_state=42)

    other_scores = []

    for row in other_refs.itertuples(index=False):
        features = spectrum_to_dict(
            row.ms2_mzs, row.ms2_normalized_intensities
        )
        norm = np.sqrt(sum(v * v for v in features.values()))
        shared = set(q_features) & set(features)

        score = (
            sum(q_features[k] * features[k] for k in shared)
            / (q_norm * norm)
            if norm else 0.0
        )
        other_scores.append(score)

    results.append({
        "inchikey14": query["inchikey14"],
        "same_identity_score": same_score,
        "best_different_score": max(other_scores),
        "same_beats_all_100": same_score > max(other_scores)
    })

results_df = pd.DataFrame(results)

print("Queries evaluated:", len(results_df))
print(
    "Same-identity score beats all 100 alternatives:",
    f'{results_df["same_beats_all_100"].sum()} / {len(results_df)}'
)
print(
    "Median same-identity score:",
    round(results_df["same_identity_score"].median(), 4)
)
print(
    "Median best-different score:",
    round(results_df["best_different_score"].median(), 4)
)

Queries evaluated: 20
Same-identity score beats all 100 alternatives: 16 / 20
Median same-identity score: 0.7342
Median best-different score: 0.1841


In [32]:
query_feature_counts = validation_sample["ms2_mzs"].map(
    lambda peaks: len(spectrum_to_dict(peaks, [1.0] * len(peaks)))
)

reference_feature_counts = refs_all["ms2_mzs"].map(
    lambda peaks: len(spectrum_to_dict(peaks, [1.0] * len(peaks)))
)

print("Validation spectra with zero features:",
      int((query_feature_counts == 0).sum()))
print("Validation spectra with features:",
      int((query_feature_counts > 0).sum()))
print("Reference identities with zero features:",
      int((reference_feature_counts == 0).sum()))
print("Reference identities with features:",
      int((reference_feature_counts > 0).sum()))
print("Median features per validation spectrum:",
      float(query_feature_counts.median()))
print("Median features per reference spectrum:",
      float(reference_feature_counts.median()))

Validation spectra with zero features: 0
Validation spectra with features: 500
Reference identities with zero features: 303
Reference identities with features: 275507
Median features per validation spectrum: 42.0
Median features per reference spectrum: 59.0


In [33]:
def fragment_cosine(features_a, features_b):
    norm_a = np.sqrt(sum(v * v for v in features_a.values()))
    norm_b = np.sqrt(sum(v * v for v in features_b.values()))

    if norm_a == 0 or norm_b == 0:
        return 0.0

    common_bins = features_a.keys() & features_b.keys()

    dot_product = sum(
        features_a[k] * features_b[k]
        for k in common_bins
    )

    return float(dot_product / (norm_a * norm_b))


# Recheck our earlier example
print(
    "Same-identity cosine:",
    round(fragment_cosine(query_features, ref_features), 4)
)

Same-identity cosine: 0.0


In [34]:
# Recreate the original validation query
example = validation_sample.iloc[0]
query_id = example["inchikey14"]

query_features = spectrum_to_dict(
    example["ms2_mzs"],
    example["ms2_normalized_intensities"]
)

# Find a separate spectrum for the same molecule
same_refs = reference_spectra[
    reference_spectra["inchikey14"] == query_id
]

ref_example = same_refs.iloc[0]

ref_features = spectrum_to_dict(
    ref_example["ms2_mzs"],
    ref_example["ms2_normalized_intensities"]
)

print("Query identity:", query_id)
print("Reference identity:", ref_example["inchikey14"])
print("Query features:", len(query_features))
print("Reference features:", len(ref_features))
print("Cosine similarity:", round(
    fragment_cosine(query_features, ref_features), 4
))

Query identity: OVMFOVNOXASTPA
Reference identity: OVMFOVNOXASTPA
Query features: 48
Reference features: 104
Cosine similarity: 0.5791


In [35]:
# Rank reference molecules for the example query
query_row_index = example.name

query_features = spectrum_to_dict(
    example["ms2_mzs"],
    example["ms2_normalized_intensities"]
)

search_refs = refs_all[
    refs_all["inchikey14"] != query_id
].copy()

# Score the first 500 reference identities as a small smoke test
small_refs = search_refs.sample(n=500, random_state=42).copy()

ranked_scores = []

for row in small_refs.itertuples(index=False):
    features = spectrum_to_dict(
        row.ms2_mzs,
        row.ms2_normalized_intensities
    )

    ranked_scores.append(
        fragment_cosine(query_features, features)
    )

small_refs["fragment_cosine"] = ranked_scores
small_refs = small_refs.sort_values(
    "fragment_cosine", ascending=False
)

print("Candidates scored:", len(small_refs))
print(
    small_refs[
        ["inchikey14", "normalized_smiles", "fragment_cosine"]
    ].head(10).to_string(index=False)
)

Candidates scored: 500
    inchikey14                                                                                                      normalized_smiles  fragment_cosine
GVQWPKKQWBUOGJ                                                                                  CC1Oc2ccccc2OC1C(=O)Nc1cccc(C(=O)O)c1         0.025372
SLTXHECURMGGES                                                                                CC(C)S(=O)(=O)CCCSc1nnc(-c2ccccc2)[nH]1         0.021798
DFYQOVFLWLMANN                                                                     COc1c(OC2OCC(O)C(O)C2O)cc2oc(-c3ccccc3)cc(=O)c2c1O         0.020765
PTZMHYXRYDXHIW                                                                             Cc1ccc(S(=O)(=O)Nc2cc3n(n2)CCN(C)C3=O)cc1F         0.017858
HOZUCTIDUAHPJW                                                                           CC(C)C1C(=O)NCCN1S(=O)(=O)c1ccc(C(F)(F)F)nc1         0.014058
SOUNSDQZADJFNN                                                         

In [36]:
# Use the query and its known identity
query_id = example["inchikey14"]

query_features = spectrum_to_dict(
    example["ms2_mzs"],
    example["ms2_normalized_intensities"]
)

# Score one spectrum per reference identity.
# refs_all contains one row per InChIKey14.
scored_refs = refs_all.copy()
scored_refs["fragment_cosine"] = [
    fragment_cosine(
        query_features,
        spectrum_to_dict(mzs, intensities)
    )
    for mzs, intensities in zip(
        scored_refs["ms2_mzs"],
        scored_refs["ms2_normalized_intensities"]
    )
]

scored_refs = scored_refs.sort_values(
    "fragment_cosine", ascending=False
).reset_index(drop=True)

matching_positions = np.flatnonzero(
    scored_refs["inchikey14"].to_numpy() == query_id
)

print("Reference identities ranked:", len(scored_refs))
print("Correct identity found:", len(matching_positions) > 0)

if len(matching_positions):
    rank = int(matching_positions[0]) + 1
    score = float(scored_refs.loc[rank - 1, "fragment_cosine"])
    print("Correct identity rank:", rank)
    print("Correct identity score:", round(score, 4))

print("\nTop 5 candidates:")
print(
    scored_refs[
        ["inchikey14", "normalized_smiles", "fragment_cosine"]
    ].head(5).to_string(index=False)
)

Reference identities ranked: 275810
Correct identity found: True
Correct identity rank: 1
Correct identity score: 0.5791

Top 5 candidates:
    inchikey14                                                                 normalized_smiles  fragment_cosine
OVMFOVNOXASTPA        O=c1cc(-c2ccc(O)cc2)oc2c(C3OC(CO)C(O)C(O)C3O)c(O)c(C3OCC(O)C(O)C3O)c(O)c12         0.579065
YCFQXQCEEPCZMO             O=c1c(-c2ccc(O)cc2)coc2c(C3OC(COC4OCC(O)C(O)C4O)C(O)C(O)C3O)c(O)ccc12         0.332841
BQJKVFXDDMQLBE COc1c2c3c4c(c(OC)c(=O)c5c(O)cc(OC)c(c6c(OC)cc(O)c(c1=O)c63)c54)C(C(C)=O)C(C)(O)C2         0.300239
QETQPQDBDULWLC          O=C1CC(c2ccc(O)c(O)c2)Oc2c1c(OC1OC(CO)C(O)C(O)C1O)cc(O)c2C1OCC(O)C(O)C1O         0.298935
ABRULANJVVJLFI                          COc1cc2oc(-c3ccc(O)cc3)cc(=O)c2c(O)c1C1OC(CO)C(O)C(O)C1O         0.269834


In [39]:
# Test full-library retrieval on 20 validation queries
test_queries = validation_sample.sample(
    n=20, random_state=123
)

reference_features = [
    spectrum_to_dict(mzs, intensities)
    for mzs, intensities in zip(
        refs_all["ms2_mzs"],
        refs_all["ms2_normalized_intensities"]
    )
]

reference_ids = refs_all["inchikey14"].to_numpy()

retrieval_results = []

for _, query in test_queries.iterrows():
    q_features = spectrum_to_dict(
        query["ms2_mzs"],
        query["ms2_normalized_intensities"]
    )

    scores = np.array([
        fragment_cosine(q_features, features)
        for features in reference_features
    ])

    # Rank with descending scores
    order = np.argsort(-scores)
    ranked_ids = reference_ids[order]
    hits = np.flatnonzero(ranked_ids == query["inchikey14"])

    rank = int(hits[0]) + 1 if len(hits) else None

    retrieval_results.append({
        "inchikey14": query["inchikey14"],
        "rank": rank,
        "top25": rank is not None and rank <= 25
    })

retrieval_df = pd.DataFrame(retrieval_results)

print("Queries evaluated:", len(retrieval_df))
print("Correct identity in top 25:",
      int(retrieval_df["top25"].sum()), "/", len(retrieval_df))
print("Median rank:",
      retrieval_df["rank"].median())
print("Ranks:")
print(retrieval_df["rank"].to_list())

Queries evaluated: 20
Correct identity in top 25: 6 / 20
Median rank: 335.0
Ranks:
[488, 202813, 185553, 1, 84, 6, 710, 95602, 182, 179966, 256912, 1, 1, 54, 5, 1331, 1, 26260, 16740, 112]


In [38]:
from pathlib import Path
from datetime import datetime

checkpoint_text = """
PROJECT: Enveda CASMI 2026 — Molecule ID From Mass Spectra

CURRENT STATUS
- Kaggle dataset loaded successfully.
- RDKit installed and working in the current session.
- Training rows: 2,539,608.
- Unique training identities (InChIKey14): 275,810.
- Test spectra: 1,213, across 400 molecule IDs.
- Candidate table and RDKit exact masses calculated.
- Fragment feature function and cosine similarity function tested.
- One known-identity retrieval ranked correctly at rank 1/275,810.
- This is a diagnostic result, NOT a competition score.

LAST EXPERIMENT
A 20-query full-reference retrieval test was started.
It may have been interrupted before completion.
Do not assume results exist unless the output was printed.

NEXT STEPS
1. Confirm whether the 20-query experiment completed.
2. Replace slow Python loops with a sparse-matrix fragment search.
3. Validate carefully with held-out spectra.
4. Distinguish known-identity retrieval from unseen-identity evaluation.
5. Build and check a valid submission CSV.

IMPORTANT NOTE
Saving this checkpoint does not save Python variables.
A restarted Kaggle session must reload data and recreate variables.
"""

checkpoint_path = Path("/kaggle/working/progress_checkpoint.txt")
checkpoint

PosixPath('/kaggle/working/progress_checkpoint.txt')

In [40]:
print("Notebook session resumed.")

for name in [
    "validation_sample",
    "reference_spectra",
    "refs_all",
    "spectrum_to_dict",
    "fragment_cosine",
    "retrieval_df",
]:
    print(f"{name}: {'available' if name in globals() else 'not available'}")

Notebook session resumed.
validation_sample: available
reference_spectra: available
refs_all: available
spectrum_to_dict: available
fragment_cosine: available
retrieval_df: available


In [41]:
import scipy
from scipy.sparse import csr_matrix

print("SciPy version:", scipy.__version__)
print("Sparse matrix support: ready")

SciPy version: 1.16.3
Sparse matrix support: ready


In [42]:
from scipy.sparse import csr_matrix
from sklearn.preprocessing import normalize
import numpy as np

# Keep this experiment small and reproducible
small_refs = refs_all.sample(n=100, random_state=42).copy()

# Convert each reference spectrum into fragment-bin features
small_feature_dicts = [
    spectrum_to_dict(mzs, intensities)
    for mzs, intensities in zip(
        small_refs["ms2_mzs"],
        small_refs["ms2_normalized_intensities"]
    )
]

# Give every observed fragment bin a column number
all_bins = sorted({
    bin_id
    for features in small_feature_dicts
    for bin_id in features
})
bin_to_col = {bin_id: col for col, bin_id in enumerate(all_bins)}

rows, cols, values = [], [], []

for row_idx, features in enumerate(small_feature_dicts):
    for bin_id, value in features.items():
        rows.append(row_idx)
        cols.append(bin_to_col[bin_id])
        values.append(value)

X_small = csr_matrix(
    (values, (rows, cols)),
    shape=(len(small_feature_dicts), len(all_bins)),
    dtype=np.float32
)

X_small = normalize(X_small, norm="l2", axis=1)

print("Matrix shape:", X_small.shape)
print("Non-zero values:", X_small.nnz)
print("Storage type:", type(X_small).__name__)

Matrix shape: (100, 8827)
Non-zero values: 17630
Storage type: csr_matrix


In [43]:
from sklearn.preprocessing import normalize
import numpy as np

# Use the first reference spectrum as our query
check_features = small_feature_dicts[0]

# Build its vector using the same fragment-bin columns
check_cols = [
    bin_to_col[k]
    for k in check_features
    if k in bin_to_col
]
check_values = [
    check_features[k]
    for k in check_features
    if k in bin_to_col
]

query_vector = csr_matrix(
    (
        np.asarray(check_values, dtype=np.float32),
        (
            np.zeros(len(check_cols), dtype=int),
            np.asarray(check_cols, dtype=int)
        )
    ),
    shape=(1, X_small.shape[1])
)

query_vector = normalize(query_vector, norm="l2", axis=1)

# Sparse matrix dot product gives cosine scores because rows are normalized
matrix_scores = (X_small @ query_vector.T).toarray().ravel()

# Compare the first row with the original function
original_score = fragment_cosine(
    small_feature_dicts[0],
    small_feature_dicts[0]
)

print("Original function (self-similarity):", round(original_score, 6))
print("Sparse matrix (self-similarity):", round(float(matrix_scores[0]), 6))
print("Maximum sparse score:", round(float(matrix_scores.max()), 6))

Original function (self-similarity): 1.0
Sparse matrix (self-similarity): 1.0
Maximum sparse score: 1.0


In [45]:
# Rebuild the original query features
example = validation_sample.iloc[0]
query_id = example["inchikey14"]

query_features = spectrum_to_dict(
    example["ms2_mzs"],
    example["ms2_normalized_intensities"]
)

# Use the separate same-identity reference spectrum
same_refs = reference_spectra[
    reference_spectra["inchikey14"] == query_id
]
ref_example = same_refs.iloc[0]

ref_features = spectrum_to_dict(
    ref_example["ms2_mzs"],
    ref_example["ms2_normalized_intensities"]
)

# Original cosine calculation
original_score = fragment_cosine(query_features, ref_features)

# Create query vector using the small matrix's bin mapping
q_cols = [
    bin_to_col[k] for k in query_features if k in bin_to_col
]
q_vals = [
    query_features[k] for k in query_features if k in bin_to_col
]

q_vector = csr_matrix(
    (
        np.asarray(q_vals, dtype=np.float32),
        (
            np.zeros(len(q_cols), dtype=int),
            np.asarray(q_cols, dtype=int)
        )
    ),
    shape=(1, X_small.shape[1])
)
q_vector = normalize(q_vector, norm="l2", axis=1)

# Matrix scores against the 100 sampled references
matrix_scores = (X_small @ q_vector.T).toarray().ravel()

print("Query identity:", query_id)
print("Original same-identity score:", round(original_score, 6))
print("Maximum score among 100 sampled references:",
      round(float(matrix_scores.max()), 6))
print("Same-identity reference included

SyntaxError: unterminated string literal (detected at line 51) (3998636393.py, line 51)

In [46]:
# Rebuild the original query features
example = validation_sample.iloc[0]
query_id = example["inchikey14"]

query_features = spectrum_to_dict(
    example["ms2_mzs"],
    example["ms2_normalized_intensities"]
)

# Find a separate reference spectrum with the same identity
same_refs = reference_spectra[
    reference_spectra["inchikey14"] == query_id
]
ref_example = same_refs.iloc[0]

ref_features = spectrum_to_dict(
    ref_example["ms2_mzs"],
    ref_example["ms2_normalized_intensities"]
)

# Original cosine calculation
original_score = fragment_cosine(query_features, ref_features)

# Convert query to the small matrix's fragment-bin columns
q_cols = [
    bin_to_col[k] for k in query_features if k in bin_to_col
]
q_vals = [
    query_features[k] for k in query_features if k in bin_to_col
]

q_vector = csr_matrix(
    (
        np.asarray(q_vals, dtype=np.float32),
        (
            np.zeros(len(q_cols), dtype=int),
            np.asarray(q_cols, dtype=int)
        )
    ),
    shape=(1, X_small.shape[1])
)
q_vector = normalize(q_vector, norm="l2", axis=1)

# Compare against the 100 sampled references
matrix_scores = (X_small @ q_vector.T).toarray().ravel()

print("Query identity:", query_id)
print("Original same-identity score:", round(original_score, 6))
print(
    "Maximum score among 100 sampled references:",
    round(float(matrix_scores.max()), 6)
)
print(
    "Same-identity reference included in sample:",
    bool((small_refs["inchikey14"] == query_id).any())
)

Query identity: OVMFOVNOXASTPA
Original same-identity score: 0.579065
Maximum score among 100 sampled references: 0.088711
Same-identity reference included in sample: False


In [47]:
import os

try:
    import psutil
    mem = psutil.virtual_memory()
    print("Total RAM (GB):", round(mem.total / 1e9, 2))
    print("Available RAM (GB):", round(mem.available / 1e9, 2))
except ImportError:
    print("psutil is not installed; we'll use a conservative chunked approach.")

Total RAM (GB): 33.66
Available RAM (GB): 17.38


In [48]:
all_reference_bins = set()
total_reference_features = 0

for mzs, intensities in zip(
    refs_all["ms2_mzs"],
    refs_all["ms2_normalized_intensities"]
):
    features = spectrum_to_dict(mzs, intensities)
    all_reference_bins.update(features.keys())
    total_reference_features += len(features)

print("Reference identities:", len(refs_all))
print("Distinct fragment bins:", len(all_reference_bins))
print("Total non-zero feature entries:", total_reference_features)
print(
    "Average non-zero features per identity:",
    round(total_reference_features / len(refs_all), 2)
)

Reference identities: 275810
Distinct fragment bins: 47501
Total non-zero feature entries: 43230905
Average non-zero features per identity: 156.74


In [49]:
from scipy.sparse import csr_matrix, vstack
from sklearn.preprocessing import normalize
import numpy as np

test_chunk = refs_all.iloc[:1000]

rows = []
cols = []
values = []

for row_idx, (mzs, intensities) in enumerate(
    zip(test_chunk["ms2_mzs"], test_chunk["ms2_normalized_intensities"])
):
    features = spectrum_to_dict(mzs, intensities)

    for bin_id, value in features.items():
        rows.append(row_idx)
        cols.append(bin_id)
        values.append(value)

X_test_chunk = csr_matrix(
    (np.array(values, dtype=np.float32),
     (np.array(rows), np.array(cols))),
    shape=(len(test_chunk), 50001),
    dtype=np.float32
)

X_test_chunk = normalize(X_test_chunk, norm="l2", axis=1)

print("Matrix shape:", X_test_chunk.shape)
print("Non-zero entries:", X_test_chunk.nnz)
print("Data type:", X_test_chunk.dtype)

Matrix shape: (1000, 50001)
Non-zero entries: 109497
Data type: float32


In [50]:
print("Matrix shape:", X_test_chunk.shape)

Matrix shape: (1000, 50001)


In [51]:
check_idx = 0

check_features = spectrum_to_dict(
    test_chunk.iloc[check_idx]["ms2_mzs"],
    test_chunk.iloc[check_idx]["ms2_normalized_intensities"]
)

query_bins = np.array(list(check_features.keys()), dtype=np.int32)
query_values = np.array(list(check_features.values()), dtype=np.float32)

query_vector = csr_matrix(
    (query_values, (np.zeros(len(query_bins), dtype=np.int32), query_bins)),
    shape=(1, 50001),
    dtype=np.float32
)

query_vector = normalize(query_vector, norm="l2", axis=1)

sparse_score = float((X_test_chunk[check_idx] @ query_vector.T).toarray()[0, 0])
original_score = fragment_cosine(check_features, check_features)

print("Sparse self-similarity:", round(sparse_score, 6))
print("Original self-similarity:", round(original_score, 6))

Sparse self-similarity: 1.0
Original self-similarity: 1.0


In [52]:
from scipy.sparse import csr_matrix, vstack
from sklearn.preprocessing import normalize
import numpy as np

chunk_size = 5000
matrix_chunks = []

for start in range(0, len(refs_all), chunk_size):
    chunk = refs_all.iloc[start:start + chunk_size]

    rows = []
    cols = []
    values = []

    for row_idx, (mzs, intensities) in enumerate(
        zip(chunk["ms2_mzs"], chunk["ms2_normalized_intensities"])
    ):
        features = spectrum_to_dict(mzs, intensities)

        for bin_id, value in features.items():
            rows.append(row_idx)
            cols.append(bin_id)
            values.append(value)

    X_chunk = csr_matrix(
        (
            np.asarray(values, dtype=np.float32),
            (
                np.asarray(rows, dtype=np.int32),
                np.asarray(cols, dtype=np.int32)
            )
        ),
        shape=(len(chunk), 50001),
        dtype=np.float32
    )

    X_chunk = normalize(X_chunk, norm="l2", axis=1)
    matrix_chunks.append(X_chunk)

    if len(matrix_chunks) % 10 == 0 or start + chunk_size >= len(refs_all):
        print(
            f"Processed {min(start + chunk_size, len(refs_all)):,}"
            f" / {len(refs_all):,} reference molecules"
        )

X_reference = vstack(matrix_chunks, format="csr", dtype=np.float32)

print("\nFull matrix shape:", X_reference.shape)
print("Non-zero entries:", X_reference.nnz)
print("Data type:", X_reference.dtype)

Processed 50,000 / 275,810 reference molecules
Processed 100,000 / 275,810 reference molecules
Processed 150,000 / 275,810 reference molecules
Processed 200,000 / 275,810 reference molecules
Processed 250,000 / 275,810 reference molecules
Processed 275,810 / 275,810 reference molecules

Full matrix shape: (275810, 50001)
Non-zero entries: 43230905
Data type: float32


In [53]:
query = validation_sample.iloc[0]

query_features = spectrum_to_dict(
    query["ms2_mzs"],
    query["ms2_normalized_intensities"]
)

query_bins = np.array(list(query_features.keys()), dtype=np.int32)
query_values = np.array(list(query_features.values()), dtype=np.float32)

query_vector = csr_matrix(
    (
        query_values,
        (np.zeros(len(query_bins), dtype=np.int32), query_bins)
    ),
    shape=(1, 50001),
    dtype=np.float32
)

query_vector = normalize(query_vector, norm="l2", axis=1)

scores = (X_reference @ query_vector.T).toarray().ravel()

top_indices = np.argsort(scores)[::-1][:25]

top25 = refs_all.iloc[top_indices][
    ["inchikey14", "normalized_smiles"]
].copy()
top25["cosine_score"] = scores[top_indices]

true_identity = query["inchikey14"]
matching_ranks = np.where(
    refs_all["inchikey14"].to_numpy()[np.argsort(scores)[::-1]] == true_identity
)[0]

true_rank = int(matching_ranks[0] + 1) if len(matching_ranks) else None

print("Query identity:", true_identity)
print("True identity rank:", true_rank)
print("\nTop 10 retrieved candidates:")
display(top25.head(10))

Query identity: OVMFOVNOXASTPA
True identity rank: 1

Top 10 retrieved candidates:


,inchikey14,normalized_smiles,cosine_score
209852,OVMFOVNOXASTPA,O=c1cc(-c2ccc(O)cc2)oc2c(C3OC(CO)C(O)C(O)C3O)c...,0.579065
262604,YCFQXQCEEPCZMO,O=c1c(-c2ccc(O)cc2)coc2c(C3OC(COC4OCC(O)C(O)C4...,0.332841
265048,BQJKVFXDDMQLBE,COc1c2c3c4c(c(OC)c(=O)c5c(O)cc(OC)c(c6c(OC)cc(...,0.300239
212265,QETQPQDBDULWLC,O=C1CC(c2ccc(O)c(O)c2)Oc2c1c(OC1OC(CO)C(O)C(O)...,0.298935
185588,ABRULANJVVJLFI,COc1cc2oc(-c3ccc(O)cc3)cc(=O)c2c(O)c1C1OC(CO)C...,0.269834
231605,YXHFXGHAELQJGK,COc1cc(-c2cc(=O)c3c(O)cc(O)c(C4OC(CO)C(O)C(O)C...,0.262467
274042,VMPAMENBRQABQP,C=C(C)c1c(O)c(O)c(C=O)c2c(O)c(-c3c(O)c4c(C=O)c...,0.262108
193063,FIAAVMJLAGNUKW,O=c1cc(-c2ccc(O)cc2)oc2c(C3OC(CO)C(O)C(O)C3O)c...,0.243310
211256,PPUYQVKBSDIARR,COc1ccc(C(=O)C(c2coc3cc(O)ccc3c2=O)C(c2ccc(O)c...,0.224380
258949,UYPPTKDFDUWVGD,O=C(Cn1c(=O)n(Cc2ccccc2)c(=O)c2ncccc21)Nc1ccc(...,0.222264


In [54]:
validation_results = []

for _, query in validation_sample.head(20).iterrows():
    query_features = spectrum_to_dict(
        query["ms2_mzs"],
        query["ms2_normalized_intensities"]
    )

    query_bins = np.array(list(query_features.keys()), dtype=np.int32)
    query_values = np.array(list(query_features.values()), dtype=np.float32)

    query_vector = csr_matrix(
        (
            query_values,
            (np.zeros(len(query_bins), dtype=np.int32), query_bins)
        ),
        shape=(1, 50001),
        dtype=np.float32
    )

    query_vector = normalize(query_vector, norm="l2", axis=1)
    scores = (X_reference @ query_vector.T).toarray().ravel()

    true_identity = query["inchikey14"]
    true_positions = np.flatnonzero(
        refs_all["inchikey14"].to_numpy() == true_identity
    )

    if len(true_positions):
        best_true_score = scores[true_positions].max()
        true_rank = int(1 + np.sum(scores > best_true_score))
    else:
        true_rank = None

    validation_results.append({
        "inchikey14": true_identity,
        "true_rank": true_rank,
        "best_true_score": (
            round(float(best_true_score), 4)
            if len(true_positions) else None
        )
    })

validation_results_df = pd.DataFrame(validation_results)

print("Queries evaluated:", len(validation_results_df))
print(
    "Correct identity in top 25:",
    (validation_results_df["true_rank"] <= 25).sum(),
    "/",
    len(validation_results_df)
)
print("Median rank:", validation_results_df["true_rank"].median())

display(validation_results_df)

Queries evaluated: 20
Correct identity in top 25: 4 / 20
Median rank: 7465.5


,inchikey14,true_rank,best_true_score
0,OVMFOVNOXASTPA,1,0.5791
1,HNXMPWWIDRRLSJ,1,0.8635
2,GPPIWOJDFFGJMA,3,0.5002
3,ARSRJFRKVXALTF,12557,0.0185
4,FTCPNPNVWBKFEE,182923,0.0000
5,PGYWEQIPESPBEW,710,0.3489
6,ORNLUXKJFDLMNT,22950,0.0317
7,SPSYWCNATOUHJM,17168,0.0015
8,BLNMYSBCIVAVFK,1,0.9985
9,ZCCUUQDIBDJBTK,42415,0.0182


In [55]:
failed_id = "FTCPNPNVWBKFEE"

failed_query = validation_sample[
    validation_sample["inchikey14"] == failed_id
].iloc[0]

failed_features = spectrum_to_dict(
    failed_query["ms2_mzs"],
    failed_query["ms2_normalized_intensities"]
)

same_identity_refs = refs_all[
    refs_all["inchikey14"] == failed_id
]

print("Identity:", failed_id)
print("Query adduct:", failed_query["adduct"])
print("Query precursor m/z:", failed_query["precursor_mz"])
print("Query fragment count:", len(failed_query["ms2_mzs"]))
print("Query usable features:", len(failed_features))
print("Reference spectra for identity:", len(same_identity_refs))

if len(same_identity_refs):
    ref_features = spectrum_to_dict(
        same_identity_refs.iloc[0]["ms2_mzs"],
        same_identity_refs.iloc[0]["ms2_normalized_intensities"]
    )
    print("First reference usable features:", len(ref_features))
    print(
        "Direct query/reference cosine:",
        round(fragment_cosine(failed_features, ref_features), 6)
    )

Identity: FTCPNPNVWBKFEE
Query adduct: [M+H]+
Query precursor m/z: 326.1055
Query fragment count: 97
Query usable features: 73
Reference spectra for identity: 1
First reference usable features: 7
Direct query/reference cosine: 0.0


In [56]:
print("QUERY")
print("First 15 fragment m/z values:")
print(np.round(failed_query["ms2_mzs"][:15], 3))
print("First 15 intensities:")
print(np.round(failed_query["ms2_normalized_intensities"][:15], 4))

failed_ref = same_identity_refs.iloc[0]

print("\nREFERENCE")
print("First 15 fragment m/z values:")
print(np.round(failed_ref["ms2_mzs"][:15], 3))
print("First 15 intensities:")
print(np.round(failed_ref["ms2_normalized_intensities"][:15], 4))

print("\nFragment m/z ranges:")
print(
    "Query:",
    (min(failed_query["ms2_mzs"]), max(failed_query["ms2_mzs"]))
)
print(
    "Reference:",
    (min(failed_ref["ms2_mzs"]), max(failed_ref["ms2_mzs"]))
)

QUERY
First 15 fragment m/z values:
[ 81.046  82.062  92.05   94.066  95.049  95.053  95.061  95.064  96.059
  96.064  96.082  97.065  97.075 108.081 109.007]
First 15 intensities:
[0.0019 0.001  0.0044 0.0018 0.0036 0.0021 0.0343 0.0028 0.0083 0.0044
 0.0022 0.0013 0.0014 0.0025 0.0045]

REFERENCE
First 15 fragment m/z values:
[326.12  348.088 348.097 349.088 349.1   350.081 350.085 357.22 ]
First 15 intensities:
[0.0387 1.     0.0242 0.3027 0.06   0.1074 0.1509 0.0406]

Fragment m/z ranges:
Query: (np.float64(81.0459), np.float64(326.1043))
Reference: (np.float64(326.1203), np.float64(357.2197))


In [57]:
print("Query metadata")
for col in [
    "ionization_mode",
    "adduct",
    "precursor_mz",
]:
    print(f"{col}: {failed_query[col]}")

print("\nReference metadata")
for col in [
    "ionization_mode",
    "adduct",
    "precursor_mz",
]:
    print(f"{col}: {failed_ref[col]}")

print("\nReference row count for this identity:", len(same_identity_refs))

Query metadata
ionization_mode: positive
adduct: [M+H]+
precursor_mz: 326.1055

Reference metadata
ionization_mode: positive
adduct: [2M+Na]+
precursor_mz: 673.1888

Reference row count for this identity: 1


In [58]:
adducts_per_identity = (
    train_spectra.groupby("inchikey14")["adduct"]
    .nunique()
)

print("Unique identities:", len(adducts_per_identity))
print(
    "Identities with multiple adduct types:",
    int((adducts_per_identity > 1).sum())
)
print(
    "Identities with only one adduct type:",
    int((adducts_per_identity == 1).sum())
)

print("\nMost common adduct labels:")
display(train_spectra["adduct"].value_counts().head(10))

Unique identities: 275810
Identities with multiple adduct types: 151289
Identities with only one adduct type: 124521

Most common adduct labels:


adduct
[M+H]+          1324585
[M-H]-           465688
[2M+Na]+         190033
[M+Na]+          178092
[2M+H]+           92041
[M+CH2O2-H]-      83188
[2M-H]-           56172
[M+NH4]+          53452
[M+Cl]-           27511
[M-H2O+H]+        21682
Name: count, dtype: int64

In [59]:
adduct_counts = (
    train_spectra.groupby("adduct")
    .agg(
        spectra=("inchikey14", "size"),
        unique_identities=("inchikey14", "nunique")
    )
    .sort_values("spectra", ascending=False)
)

display(adduct_counts.head(15))

,spectra,unique_identities
adduct,,
[M+H]+,1324585,243970
[M-H]-,465688,98485
[2M+Na]+,190033,55466
[M+Na]+,178092,25112
[2M+H]+,92041,25705
[M+CH2O2-H]-,83188,20723
[2M-H]-,56172,15994
[M+NH4]+,53452,8890
[M+Cl]-,27511,7148


In [60]:
reference_counts = (
    train_spectra.groupby(["inchikey14", "adduct"])
    .size()
)

print("All training spectra:", len(train_spectra))
print(
    "Unique identity-adduct pairs:",
    len(reference_counts)
)
print(
    "Pairs with 2+ spectra:",
    int((reference_counts >= 2).sum())
)
print(
    "Spectra retained if we keep up to 3 per identity-adduct pair:",
    int(reference_counts.clip(upper=3).sum())
)

All training spectra: 2539608
Unique identity-adduct pairs: 519426
Pairs with 2+ spectra: 464902
Spectra retained if we keep up to 3 per identity-adduct pair: 1380008


In [61]:
failed_id = "FTCPNPNVWBKFEE"

failed_refs = (
    train_spectra[train_spectra["inchikey14"] == failed_id]
    .groupby("adduct", group_keys=False)
    .head(3)
    .copy()
)

failed_query = validation_sample[
    validation_sample["inchikey14"] == failed_id
].iloc[0]

query_features = spectrum_to_dict(
    failed_query["ms2_mzs"],
    failed_query["ms2_normalized_intensities"]
)

failed_refs["fragment_features"] = [
    [
        fragment_cosine(
            query_features,
            spectrum_to_dict(mzs, intensities)
        )
    ][0]
    for mzs, intensities in zip(
        failed_refs["ms2_mzs"],
        failed_refs["ms2_normalized_intensities"]
    )
]

display(
    failed_refs[
        ["adduct", "precursor_mz", "fragment_features"]
    ].sort_values("fragment_features", ascending=False)
)

,adduct,precursor_mz,fragment_features
228705,[M+H]+,326.1055,1.000000
228706,[M+H]+,326.1056,0.738428
228707,[M+H]+,326.1056,0.668845
228709,[M-H]-,324.0912,0.155554
228711,[M-H]-,324.0912,0.139391
228710,[M-H]-,324.0912,0.104703
228703,[2M+Na]+,673.1888,0.000000
228704,[2M+Na]+,673.1888,0.000000


In [62]:
reference_multi = (
    train_spectra
    .groupby(["inchikey14", "adduct"], sort=False, group_keys=False)
    .head(3)
    .reset_index(drop=True)
)

print("Reference spectra:", len(reference_multi))
print("Unique molecules:", reference_multi["inchikey14"].nunique())
print("Unique molecule-adduct pairs:",
      reference_multi[["inchikey14", "adduct"]].drop_duplicates().shape[0])

Reference spectra: 1380008
Unique molecules: 275810
Unique molecule-adduct pairs: 519426


In [63]:
import psutil

memory = psutil.virtual_memory()

print("Total RAM (GB):", round(memory.total / 1e9, 2))
print("Available RAM (GB):", round(memory.available / 1e9, 2))
print("Memory currently used (%):", memory.percent)

Total RAM (GB): 33.66
Available RAM (GB): 17.34
Memory currently used (%): 48.5


In [64]:
from scipy.sparse import csr_matrix, vstack
from sklearn.preprocessing import normalize
import numpy as np

chunk_size = 5000
matrix_chunks_multi = []

for start in range(0, len(reference_multi), chunk_size):
    chunk = reference_multi.iloc[start:start + chunk_size]

    rows = []
    cols = []
    values = []

    for row_idx, (mzs, intensities) in enumerate(
        zip(chunk["ms2_mzs"], chunk["ms2_normalized_intensities"])
    ):
        features = spectrum_to_dict(mzs, intensities)

        for bin_id, value in features.items():
            rows.append(row_idx)
            cols.append(bin_id)
            values.append(value)

    X_chunk = csr_matrix(
        (
            np.asarray(values, dtype=np.float32),
            (
                np.asarray(rows, dtype=np.int32),
                np.asarray(cols, dtype=np.int32)
            )
        ),
        shape=(len(chunk), 50001),
        dtype=np.float32
    )

    X_chunk = normalize(X_chunk, norm="l2", axis=1)
    matrix_chunks_multi.append(X_chunk)

    if (start + chunk_size) % 50000 == 0 or start + chunk_size >= len(reference_multi):
        print(
            f"Processed {min(start + chunk_size, len(reference_multi)):,}"
            f" / {len(reference_multi):,} spectra"
        )

X_reference_multi = vstack(
    matrix_chunks_multi, format="csr", dtype=np.float32
)

print("\nExpanded matrix shape:", X_reference_multi.shape)
print("Non-zero entries:", X_reference_multi.nnz)
print("Data type:", X_reference_multi.dtype)

Processed 50,000 / 1,380,008 spectra
Processed 100,000 / 1,380,008 spectra
Processed 150,000 / 1,380,008 spectra
Processed 200,000 / 1,380,008 spectra
Processed 250,000 / 1,380,008 spectra
Processed 300,000 / 1,380,008 spectra
Processed 350,000 / 1,380,008 spectra
Processed 400,000 / 1,380,008 spectra
Processed 450,000 / 1,380,008 spectra
Processed 500,000 / 1,380,008 spectra
Processed 550,000 / 1,380,008 spectra
Processed 600,000 / 1,380,008 spectra
Processed 650,000 / 1,380,008 spectra
Processed 700,000 / 1,380,008 spectra
Processed 750,000 / 1,380,008 spectra
Processed 800,000 / 1,380,008 spectra
Processed 850,000 / 1,380,008 spectra
Processed 900,000 / 1,380,008 spectra
Processed 950,000 / 1,380,008 spectra
Processed 1,000,000 / 1,380,008 spectra
Processed 1,050,000 / 1,380,008 spectra
Processed 1,100,000 / 1,380,008 spectra
Processed 1,150,000 / 1,380,008 spectra
Processed 1,200,000 / 1,380,008 spectra
Processed 1,250,000 / 1,380,008 spectra
Processed 1,300,000 / 1,380,008 spectra

In [65]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.preprocessing import normalize

# Use the same small validation sample as before
test_queries = validation_sample.head(20).copy()

# Map every reference spectrum to its molecule identity
reference_labels = reference_multi["inchikey14"].to_numpy()

results_multi = []

for _, query in test_queries.iterrows():
    query_features = spectrum_to_dict(
        query["ms2_mzs"],
        query["ms2_normalized_intensities"]
    )

    if not query_features:
        results_multi.append({
            "inchikey14": query["inchikey14"],
            "rank": None,
            "top25": False
        })
        continue

    cols = np.array(list(query_features.keys()), dtype=np.int32)
    values = np.array(list(query_features.values()), dtype=np.float32)
    rows = np.zeros(len(cols), dtype=np.int32)

    X_query = csr_matrix(
        (values, (rows, cols)),
        shape=(1, 50001),
        dtype=np.float32
    )
    X_query = normalize(X_query, norm="l2", axis=1)

    # Similarity to every reference spectrum
    scores = (X_reference_multi @ X_query.T).toarray().ravel()

    # Keep the highest similarity for each molecule
    molecule_scores = (
        pd.DataFrame({
            "inchikey14": reference_labels,
            "score": scores
        })
        .groupby("inchikey14", sort=False)["score"]
        .max()
        .sort_values(ascending=False)
    )

    true_id = query["inchikey14"]
    rank = (
        int(np.where(molecule_scores.index.to_numpy() == true_id)[0][0]) + 1
        if true_id in molecule_scores.index
        else None
    )

    results_multi.append({
        "inchikey14": true_id,
        "rank": rank,
        "top25": rank is not None and rank <= 25
    })

results_multi_df = pd.DataFrame(results_multi)

print("Queries evaluated:", len(results_multi_df))
print("True identity in top 25:", results_multi_df["top25"].sum(), "/", len(results_multi_df))
print("Top-25 hit rate:", round(results_multi_df["top25"].mean(), 3))
print("Median rank:", results_multi_df["rank"].median())
display(results_multi_df)

Queries evaluated: 20
True identity in top 25: 20 / 20
Top-25 hit rate: 1.0
Median rank: 1.0


,inchikey14,rank,top25
0,OVMFOVNOXASTPA,1,True
1,HNXMPWWIDRRLSJ,1,True
2,GPPIWOJDFFGJMA,1,True
3,ARSRJFRKVXALTF,1,True
4,FTCPNPNVWBKFEE,1,True
5,PGYWEQIPESPBEW,1,True
6,ORNLUXKJFDLMNT,1,True
7,SPSYWCNATOUHJM,2,True
8,BLNMYSBCIVAVFK,1,True
9,ZCCUUQDIBDJBTK,1,True


In [66]:
# Check whether the validation query rows also appear in the reference library
validation_index_set = set(validation_indices)

leaked_rows = reference_multi.index.intersection(validation_index_set)

print("Validation rows:", len(validation_indices))
print("Validation indices found in reference_multi:", len(leaked_rows))
print("Reference spectra:", len(reference_multi))

Validation rows: 500
Validation indices found in reference_multi: 283
Reference spectra: 1380008


In [67]:
reference_multi_clean = (
    reference_spectra
    .groupby(["inchikey14", "adduct"], sort=False, group_keys=False)
    .head(3)
    .reset_index(drop=True)
)

print("Clean reference spectra:", len(reference_multi_clean))
print("Distinct molecules:", reference_multi_clean["inchikey14"].nunique())
print(
    "Validation rows excluded:",
    len(validation_sample),
    "held-out query rows"
)

Clean reference spectra: 1379915
Distinct molecules: 275810
Validation rows excluded: 500 held-out query rows


In [68]:
from scipy.sparse import csr_matrix, vstack
from sklearn.preprocessing import normalize
import numpy as np

chunk_size = 5000
matrix_chunks_clean = []

for start in range(0, len(reference_multi_clean), chunk_size):
    chunk = reference_multi_clean.iloc[start:start + chunk_size]
    rows, cols, values = [], [], []

    for row_idx, (mzs, intensities) in enumerate(
        zip(chunk["ms2_mzs"], chunk["ms2_normalized_intensities"])
    ):
        features = spectrum_to_dict(mzs, intensities)
        for bin_id, value in features.items():
            rows.append(row_idx)
            cols.append(bin_id)
            values.append(value)

    X_chunk = csr_matrix(
        (
            np.asarray(values, dtype=np.float32),
            (
                np.asarray(rows, dtype=np.int32),
                np.asarray(cols, dtype=np.int32)
            )
        ),
        shape=(len(chunk), 50001),
        dtype=np.float32
    )
    X_chunk = normalize(X_chunk, norm="l2", axis=1)
    matrix_chunks_clean.append(X_chunk)

    processed = min(start + chunk_size, len(reference_multi_clean))
    if processed % 50000 == 0 or processed == len(reference_multi_clean):
        print(f"Processed {processed:,} / {len(reference_multi_clean):,}")

X_reference_clean = vstack(matrix_chunks_clean, format="csr", dtype=np.float32)

print("Clean matrix shape:", X_reference_clean.shape)
print("Non-zero entries:", X_reference_clean.nnz)
print("Data type:", X_reference_clean.dtype)

Processed 50,000 / 1,379,915
Processed 100,000 / 1,379,915
Processed 150,000 / 1,379,915
Processed 200,000 / 1,379,915
Processed 250,000 / 1,379,915
Processed 300,000 / 1,379,915
Processed 350,000 / 1,379,915
Processed 400,000 / 1,379,915
Processed 450,000 / 1,379,915
Processed 500,000 / 1,379,915
Processed 550,000 / 1,379,915
Processed 600,000 / 1,379,915
Processed 650,000 / 1,379,915
Processed 700,000 / 1,379,915
Processed 750,000 / 1,379,915
Processed 800,000 / 1,379,915
Processed 850,000 / 1,379,915
Processed 900,000 / 1,379,915
Processed 950,000 / 1,379,915
Processed 1,000,000 / 1,379,915
Processed 1,050,000 / 1,379,915
Processed 1,100,000 / 1,379,915
Processed 1,150,000 / 1,379,915
Processed 1,200,000 / 1,379,915
Processed 1,250,000 / 1,379,915
Processed 1,300,000 / 1,379,915
Processed 1,350,000 / 1,379,915
Processed 1,379,915 / 1,379,915
Clean matrix shape: (1379915, 50001)
Non-zero entries: 224878601
Data type: float32
